# Reuse-Only Final All-Model Ensemble

This notebook **does not retrain Rotation, SimCLR, or any fusion model**.

It reuses the saved output of:

```text
last-part-of-the-previous-one
```

and loads the already-created validation/test probability files for:

- SimCLR multi-seed without TTA
- SimCLR multi-seed with validation-selected TTA
- Standalone Rotation
- FusionConcat
- FusionGated V2
- FusionGated V3
- Residual Fusion
- Previous fixed weighted ensemble, when available

It then performs:

1. Validation-only temperature calibration
2. Constrained all-model weighted search
3. Validation-fold logistic stacking
4. No-TTA SimCLR + stacking blend
5. Small validation-only class-bias correction
6. Frozen internal-test evaluation
7. Explicit comparison against single and fusion models

No original model-training cell is included.


In [1]:
# ============================================================
# CELL 01: IMPORTS AND OUTPUT DIRECTORY
# ============================================================

import json
import math
import shutil
from collections import OrderedDict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

RANDOM_STATE = 42
RNG = np.random.default_rng(RANDOM_STATE)

KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path(
    "/kaggle/working/reuse_only_all_model_ensemble"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREFERRED_NOTEBOOK_SLUG = "last-part-of-the-previous-one"

CLASS_NAMES = [
    "MEL",
    "NV",
    "BCC",
    "AK",
    "BKL",
    "DF",
    "VASC",
    "SCC",
]
NUM_CLASSES = len(CLASS_NAMES)

print("=" * 88)
print("REUSE-ONLY ALL-MODEL ENSEMBLE")
print("=" * 88)
print("Output directory:", OUTPUT_DIR)


REUSE-ONLY ALL-MODEL ENSEMBLE
Output directory: /kaggle/working/reuse_only_all_model_ensemble


In [2]:
# ============================================================
# CELL 02: LOCATE THE SAVED PREVIOUS-NOTEBOOK OUTPUT
# ============================================================

MARKER_RELATIVE_PATH = (
    "simclr_dominant_weighted_ensemble_v2/"
    "best_weights.json"
)

marker_matches = sorted(
    KAGGLE_INPUT.rglob(MARKER_RELATIVE_PATH)
)

if not marker_matches:
    raise FileNotFoundError(
        "The saved output of 'last-part-of-the-previous-one' "
        "was not found. Attach it through Kaggle Add Input."
    )

print("Framework markers found:")
for index, marker in enumerate(marker_matches):
    print(f"[{index}] {marker}")

preferred_matches = [
    marker
    for marker in marker_matches
    if PREFERRED_NOTEBOOK_SLUG in marker.parts
]

if not preferred_matches:
    raise FileNotFoundError(
        "Other framework outputs were found, but the required "
        "'last-part-of-the-previous-one' output was not attached."
    )

SELECTED_MARKER = sorted(
    preferred_matches,
    key=lambda path: (len(path.parts), str(path)),
)[0]

PIPELINE_ROOT = SELECTED_MARKER.parents[1]

print("\nSelected marker:")
print(SELECTED_MARKER)

print("\nPipeline root:")
print(PIPELINE_ROOT)


Framework markers found:
[0] /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/simclr_dominant_weighted_ensemble_v2/best_weights.json

Selected marker:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/simclr_dominant_weighted_ensemble_v2/best_weights.json

Pipeline root:
/kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike


In [3]:
# ============================================================
# CELL 03: LOAD ALL SAVED INTERNAL VALIDATION/TEST PROBABILITIES
# ============================================================

def normalize_probabilities(probabilities, epsilon=1e-12):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    if probabilities.ndim != 2:
        raise ValueError(
            "Probability arrays must be two-dimensional. "
            f"Received {probabilities.shape}."
        )

    probabilities = np.nan_to_num(
        probabilities,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    probabilities = np.clip(
        probabilities,
        epsilon,
        None,
    )
    probabilities /= probabilities.sum(
        axis=1,
        keepdims=True,
    )

    return probabilities.astype(np.float32)


def probability_columns(dataframe):
    standard_columns = [
        column
        for column in dataframe.columns
        if column.startswith("prob_class_")
    ]

    if standard_columns:
        return sorted(
            standard_columns,
            key=lambda column: int(
                column.rsplit("_", 1)[1]
            ),
        )

    rotation_columns = [
        column
        for column in dataframe.columns
        if column.startswith("prob_")
        and column.split("_")[1].isdigit()
    ]

    if rotation_columns:
        return sorted(
            rotation_columns,
            key=lambda column: int(
                column.split("_")[1]
            ),
        )

    raise ValueError(
        "No probability columns were found."
    )


def label_column(dataframe):
    for candidate in [
        "y_true",
        "true_label",
        "label",
    ]:
        if candidate in dataframe.columns:
            return candidate

    raise ValueError(
        "No true-label column was found."
    )


def load_prediction_csv(csv_path):
    dataframe = pd.read_csv(csv_path)

    label_name = label_column(dataframe)
    prob_columns = probability_columns(dataframe)

    y_true = dataframe[
        label_name
    ].to_numpy(dtype=np.int32)

    y_prob = normalize_probabilities(
        dataframe[
            prob_columns
        ].to_numpy(dtype=np.float32)
    )

    return y_true, y_prob


MODEL_PATHS = OrderedDict(
    [
        (
            "SimCLR_NoTTA",
            {
                "type": "npz",
                "path": (
                    PIPELINE_ROOT
                    / "simclr_finetune_multiseed_bacc_v2"
                    / "multiseed_probabilities_no_tta.npz"
                ),
            },
        ),
        (
            "SimCLR_SelectedTTA",
            {
                "type": "npz",
                "path": (
                    PIPELINE_ROOT
                    / "simclr_finetune_multiseed_bacc_v2"
                    / "tta_selection"
                    / "selected_multiseed_tta_probabilities.npz"
                ),
            },
        ),
        (
            "Rotation",
            {
                "type": "csv",
                "val": (
                    PIPELINE_ROOT
                    / "extended_imagenet_rotation_finetune"
                    / "imagenet_rotation_val_predictions.csv"
                ),
                "test": (
                    PIPELINE_ROOT
                    / "extended_imagenet_rotation_finetune"
                    / "imagenet_rotation_test_predictions.csv"
                ),
            },
        ),
        (
            "FusionConcat",
            {
                "type": "csv",
                "val": (
                    PIPELINE_ROOT
                    / "fusion_concat_multiseed_rep_v2"
                    / "full_evaluation"
                    / "val"
                    / "val_predictions.csv"
                ),
                "test": (
                    PIPELINE_ROOT
                    / "fusion_concat_multiseed_rep_v2"
                    / "full_evaluation"
                    / "test"
                    / "test_predictions.csv"
                ),
            },
        ),
        (
            "FusionGatedV2",
            {
                "type": "csv",
                "val": (
                    PIPELINE_ROOT
                    / "fusion_gated_v2_multiseed_rep_v2"
                    / "full_evaluation"
                    / "val"
                    / "val_predictions.csv"
                ),
                "test": (
                    PIPELINE_ROOT
                    / "fusion_gated_v2_multiseed_rep_v2"
                    / "full_evaluation"
                    / "test"
                    / "test_predictions.csv"
                ),
            },
        ),
        (
            "FusionGatedV3",
            {
                "type": "csv",
                "val": (
                    PIPELINE_ROOT
                    / "fusion_gated_v3_multiseed_rep_v2"
                    / "full_evaluation"
                    / "val"
                    / "val_predictions.csv"
                ),
                "test": (
                    PIPELINE_ROOT
                    / "fusion_gated_v3_multiseed_rep_v2"
                    / "full_evaluation"
                    / "test"
                    / "test_predictions.csv"
                ),
            },
        ),
        (
            "ResidualFusion",
            {
                "type": "npz",
                "path": (
                    PIPELINE_ROOT
                    / "residual_simclr_rotation_fusion_v1"
                    / "multiseed_probabilities.npz"
                ),
            },
        ),
    ]
)

VAL_PROBABILITIES = OrderedDict()
TEST_PROBABILITIES = OrderedDict()

Y_VAL = None
Y_TEST = None
availability_rows = []

for model_name, specification in MODEL_PATHS.items():
    available = False
    reason = ""

    try:
        if specification["type"] == "npz":
            artifact_path = specification["path"]

            if not artifact_path.exists():
                raise FileNotFoundError(
                    str(artifact_path)
                )

            with np.load(artifact_path) as saved:
                val_prob = normalize_probabilities(
                    saved["val_prob"]
                )
                test_prob = normalize_probabilities(
                    saved["test_prob"]
                )
                y_val = np.asarray(
                    saved["y_val"],
                    dtype=np.int32,
                )
                y_test = np.asarray(
                    saved["y_test"],
                    dtype=np.int32,
                )

        else:
            val_path = specification["val"]
            test_path = specification["test"]

            if not val_path.exists():
                raise FileNotFoundError(
                    str(val_path)
                )

            if not test_path.exists():
                raise FileNotFoundError(
                    str(test_path)
                )

            y_val, val_prob = load_prediction_csv(
                val_path
            )
            y_test, test_prob = load_prediction_csv(
                test_path
            )

        if Y_VAL is None:
            Y_VAL = y_val
            Y_TEST = y_test
        else:
            if not np.array_equal(
                Y_VAL,
                y_val,
            ):
                raise ValueError(
                    f"{model_name} validation labels are not aligned."
                )

            if not np.array_equal(
                Y_TEST,
                y_test,
            ):
                raise ValueError(
                    f"{model_name} test labels are not aligned."
                )

        VAL_PROBABILITIES[
            model_name
        ] = val_prob

        TEST_PROBABILITIES[
            model_name
        ] = test_prob

        available = True
        reason = "Loaded successfully"

    except Exception as error:
        reason = str(error)

    availability_rows.append(
        {
            "model": model_name,
            "available": available,
            "reason": reason,
        }
    )

AVAILABILITY_DF = pd.DataFrame(
    availability_rows
)

display(AVAILABILITY_DF)

required_models = [
    "SimCLR_NoTTA",
    "Rotation",
    "FusionConcat",
]

missing_required = [
    model_name
    for model_name in required_models
    if model_name not in VAL_PROBABILITIES
]

if missing_required:
    raise FileNotFoundError(
        "Required saved model predictions are missing: "
        f"{missing_required}"
    )

MODEL_NAMES = list(
    VAL_PROBABILITIES.keys()
)

anchor_shape = VAL_PROBABILITIES[
    "SimCLR_NoTTA"
].shape

for model_name in MODEL_NAMES:
    if (
        VAL_PROBABILITIES[
            model_name
        ].shape
        != anchor_shape
    ):
        raise ValueError(
            f"{model_name} validation shape does not match."
        )

    if (
        TEST_PROBABILITIES[
            model_name
        ].shape[1]
        != NUM_CLASSES
    ):
        raise ValueError(
            f"{model_name} class count does not match."
        )

print("\nLoaded models:")
print(MODEL_NAMES)

print("\nValidation shape:")
print(anchor_shape)

print("\nTest shape:")
print(
    TEST_PROBABILITIES[
        "SimCLR_NoTTA"
    ].shape
)

AVAILABILITY_DF.to_csv(
    OUTPUT_DIR
    / "loaded_saved_models.csv",
    index=False,
)


                model  available               reason
0        SimCLR_NoTTA       True  Loaded successfully
1  SimCLR_SelectedTTA       True  Loaded successfully
2            Rotation       True  Loaded successfully
3        FusionConcat       True  Loaded successfully
4       FusionGatedV2       True  Loaded successfully
5       FusionGatedV3       True  Loaded successfully
6      ResidualFusion       True  Loaded successfully


Loaded models:
['SimCLR_NoTTA', 'SimCLR_SelectedTTA', 'Rotation', 'FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']

Validation shape:
(2533, 8)

Test shape:
(2534, 8)


In [4]:
# ============================================================
# CELL 04: METRICS AND VALIDATION-FOLD DEFINITIONS
# ============================================================

MODEL_CATEGORY = {
    "SimCLR_NoTTA": "Single",
    "SimCLR_SelectedTTA": "Single",
    "Rotation": "Single",
    "FusionConcat": "Fusion",
    "FusionGatedV2": "Fusion",
    "FusionGatedV3": "Fusion",
    "ResidualFusion": "Fusion",
}


def macro_specificity(confusion):
    total = confusion.sum()
    values = []

    for class_id in range(
        confusion.shape[0]
    ):
        true_positive = confusion[
            class_id,
            class_id,
        ]
        false_negative = (
            confusion[
                class_id,
                :,
            ].sum()
            - true_positive
        )
        false_positive = (
            confusion[
                :,
                class_id,
            ].sum()
            - true_positive
        )
        true_negative = (
            total
            - true_positive
            - false_negative
            - false_positive
        )

        denominator = (
            true_negative
            + false_positive
        )

        values.append(
            true_negative / denominator
            if denominator > 0
            else 0.0
        )

    return float(np.mean(values))


def evaluate_probabilities(
    y_true,
    y_prob,
    model_name,
    split_name,
    category,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    )
    y_prob = normalize_probabilities(
        y_prob
    )
    y_pred = np.argmax(
        y_prob,
        axis=1,
    )

    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
    )

    y_binary = label_binarize(
        y_true,
        classes=np.arange(NUM_CLASSES),
    )

    try:
        auc = roc_auc_score(
            y_binary,
            y_prob,
            average="macro",
            multi_class="ovr",
        )
    except Exception:
        auc = float("nan")

    return {
        "model": model_name,
        "category": category,
        "split": split_name,
        "n_samples": int(
            len(y_true)
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "bacc": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "macro_specificity": (
            macro_specificity(
                confusion
            )
        ),
        "auc_macro_ovr": float(auc),
        "y_pred": y_pred,
        "confusion_matrix": confusion,
        "classification_report": (
            classification_report(
                y_true,
                y_pred,
                labels=np.arange(
                    NUM_CLASSES
                ),
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0,
            )
        ),
    }


FOLDS = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    ).split(
        np.zeros(len(Y_VAL)),
        Y_VAL,
    )
)

ANCHOR_NAME = "SimCLR_NoTTA"
ANCHOR_VAL_PROB = VAL_PROBABILITIES[
    ANCHOR_NAME
]
ANCHOR_VAL_PRED = np.argmax(
    ANCHOR_VAL_PROB,
    axis=1,
)

ANCHOR_FOLD_BACC = np.asarray(
    [
        balanced_accuracy_score(
            Y_VAL[fold_indices],
            ANCHOR_VAL_PRED[
                fold_indices
            ],
        )
        for _, fold_indices
        in FOLDS
    ]
)

ANCHOR_FULL_BACC = float(
    balanced_accuracy_score(
        Y_VAL,
        ANCHOR_VAL_PRED,
    )
)

print(
    "No-TTA SimCLR validation BACC:",
    ANCHOR_FULL_BACC,
)
print(
    "Validation-fold BACCs:",
    ANCHOR_FOLD_BACC.tolist(),
)


No-TTA SimCLR validation BACC: 0.7878464645725649
Validation-fold BACCs: [0.8433004021183262, 0.7643060057817239, 0.7472308387678037, 0.8250076528529836, 0.7652551153801137]


In [5]:
# ============================================================
# CELL 05: TEMPERATURE CALIBRATION USING VALIDATION ONLY
# ============================================================

TEMPERATURE_CANDIDATES = [
    0.60,
    0.70,
    0.80,
    0.90,
    1.00,
    1.10,
    1.20,
    1.30,
    1.50,
    1.75,
    2.00,
]


def temperature_scale(
    probabilities,
    temperature,
):
    probabilities = normalize_probabilities(
        probabilities
    )

    logits = np.log(
        np.clip(
            probabilities,
            1e-12,
            1.0,
        )
    )
    logits /= float(temperature)
    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    return normalize_probabilities(
        np.exp(logits)
    )


TEMPERATURES = {}
CALIBRATED_VAL = OrderedDict()
CALIBRATED_TEST = OrderedDict()
temperature_rows = []

for model_name in MODEL_NAMES:
    candidate_rows = []

    for temperature in TEMPERATURE_CANDIDATES:
        calibrated = temperature_scale(
            VAL_PROBABILITIES[
                model_name
            ],
            temperature,
        )

        candidate_rows.append(
            {
                "temperature": temperature,
                "nll": log_loss(
                    Y_VAL,
                    calibrated,
                    labels=np.arange(
                        NUM_CLASSES
                    ),
                ),
            }
        )

    candidate_df = pd.DataFrame(
        candidate_rows
    )

    best_row = candidate_df.sort_values(
        [
            "nll",
            "temperature",
        ],
        ascending=[
            True,
            True,
        ],
    ).iloc[0]

    best_temperature = float(
        best_row["temperature"]
    )

    TEMPERATURES[
        model_name
    ] = best_temperature

    CALIBRATED_VAL[
        model_name
    ] = temperature_scale(
        VAL_PROBABILITIES[
            model_name
        ],
        best_temperature,
    )

    CALIBRATED_TEST[
        model_name
    ] = temperature_scale(
        TEST_PROBABILITIES[
            model_name
        ],
        best_temperature,
    )

    temperature_rows.append(
        {
            "model": model_name,
            "temperature": best_temperature,
            "validation_nll": float(
                best_row["nll"]
            ),
        }
    )

TEMPERATURE_DF = pd.DataFrame(
    temperature_rows
)

display(TEMPERATURE_DF)

TEMPERATURE_DF.to_csv(
    OUTPUT_DIR
    / "temperature_scaling.csv",
    index=False,
)


                model  temperature  validation_nll
0        SimCLR_NoTTA         1.30        0.476873
1  SimCLR_SelectedTTA         1.30        0.471121
2            Rotation         1.75        0.544185
3        FusionConcat         1.75        0.613571
4       FusionGatedV2         1.75        0.572133
5       FusionGatedV3         2.00        0.603812
6      ResidualFusion         2.00        0.561596

In [6]:
# ============================================================
# CELL 06: CONSTRAINED ALL-MODEL WEIGHTED SEARCH
# ============================================================

RANDOM_WEIGHT_CANDIDATES = 50000
WEIGHT_CHUNK_SIZE = 128
ANCHOR_WEIGHT_MIN = 0.35
ANCHOR_WEIGHT_MAX = 0.95
MIN_FOLD_WINS = 4


def bacc_many(
    prediction_matrix,
    y_true,
    indices=None,
):
    prediction_matrix = np.asarray(
        prediction_matrix
    )
    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    )

    if indices is not None:
        prediction_matrix = (
            prediction_matrix[
                :,
                indices,
            ]
        )
        y_true = y_true[indices]

    recalls = []

    for class_id in np.unique(
        y_true
    ):
        mask = y_true == class_id

        recalls.append(
            np.mean(
                prediction_matrix[
                    :,
                    mask,
                ]
                == class_id,
                axis=1,
            )
        )

    return np.mean(
        np.stack(
            recalls,
            axis=1,
        ),
        axis=1,
    )


def generate_weight_candidates():
    n_models = len(MODEL_NAMES)
    anchor_index = MODEL_NAMES.index(
        ANCHOR_NAME
    )

    candidates = []

    # Deterministic pairwise blends.
    for auxiliary_index in range(
        n_models
    ):
        if auxiliary_index == anchor_index:
            continue

        for auxiliary_weight in np.arange(
            0.05,
            0.66,
            0.05,
        ):
            weights = np.zeros(
                n_models,
                dtype=np.float64,
            )
            weights[
                anchor_index
            ] = 1.0 - auxiliary_weight
            weights[
                auxiliary_index
            ] = auxiliary_weight

            if (
                ANCHOR_WEIGHT_MIN
                <= weights[anchor_index]
                <= ANCHOR_WEIGHT_MAX
            ):
                candidates.append(
                    weights
                )

    # Random constrained Dirichlet blends.
    anchor_weights = RNG.uniform(
        ANCHOR_WEIGHT_MIN,
        ANCHOR_WEIGHT_MAX,
        size=RANDOM_WEIGHT_CANDIDATES,
    )

    auxiliary_indices = [
        index
        for index in range(
            n_models
        )
        if index != anchor_index
    ]

    rest = RNG.dirichlet(
        np.full(
            len(auxiliary_indices),
            0.70,
        ),
        size=RANDOM_WEIGHT_CANDIDATES,
    )

    random_weights = np.zeros(
        (
            RANDOM_WEIGHT_CANDIDATES,
            n_models,
        ),
        dtype=np.float64,
    )

    random_weights[
        :,
        anchor_index,
    ] = anchor_weights

    random_weights[
        :,
        auxiliary_indices,
    ] = (
        rest
        * (
            1.0
            - anchor_weights[
                :,
                None,
            ]
        )
    )

    candidates.extend(
        random_weights
    )

    candidates = np.asarray(
        candidates,
        dtype=np.float32,
    )

    active_models = np.sum(
        candidates > 1e-4,
        axis=1,
    )
    candidates = candidates[
        active_models >= 2
    ]

    rounded = np.round(
        candidates,
        6,
    )
    _, unique_indices = np.unique(
        rounded,
        axis=0,
        return_index=True,
    )

    return candidates[
        np.sort(unique_indices)
    ]


VAL_STACK = np.stack(
    [
        CALIBRATED_VAL[
            model_name
        ]
        for model_name in MODEL_NAMES
    ],
    axis=0,
)

TEST_STACK = np.stack(
    [
        CALIBRATED_TEST[
            model_name
        ]
        for model_name in MODEL_NAMES
    ],
    axis=0,
)

WEIGHT_CANDIDATES = (
    generate_weight_candidates()
)

search_parts = []

for start in range(
    0,
    len(WEIGHT_CANDIDATES),
    WEIGHT_CHUNK_SIZE,
):
    stop = min(
        start + WEIGHT_CHUNK_SIZE,
        len(WEIGHT_CANDIDATES),
    )

    weights = WEIGHT_CANDIDATES[
        start:stop
    ]

    probabilities = np.einsum(
        "bm,mnc->bnc",
        weights,
        VAL_STACK,
        optimize=True,
    )

    predictions = np.argmax(
        probabilities,
        axis=2,
    )

    full_bacc = bacc_many(
        predictions,
        Y_VAL,
    )

    fold_scores = np.stack(
        [
            bacc_many(
                predictions,
                Y_VAL,
                fold_indices,
            )
            for _, fold_indices
            in FOLDS
        ],
        axis=1,
    )

    part = pd.DataFrame(
        {
            "candidate_index": np.arange(
                start,
                stop,
            ),
            "cv_mean_bacc": fold_scores.mean(
                axis=1
            ),
            "cv_std_bacc": fold_scores.std(
                axis=1
            ),
            "full_val_bacc": full_bacc,
            "full_val_gain_over_no_tta": (
                full_bacc
                - ANCHOR_FULL_BACC
            ),
            "fold_wins_over_no_tta": np.sum(
                fold_scores
                > (
                    ANCHOR_FOLD_BACC[
                        None,
                        :,
                    ]
                    + 1e-12
                ),
                axis=1,
            ),
            "anchor_weight": weights[
                :,
                MODEL_NAMES.index(
                    ANCHOR_NAME
                ),
            ],
        }
    )

    for model_index, model_name in enumerate(
        MODEL_NAMES
    ):
        part[
            f"weight__{model_name}"
        ] = weights[
            :,
            model_index,
        ]

    search_parts.append(part)

WEIGHT_SEARCH_DF = pd.concat(
    search_parts,
    ignore_index=True,
).sort_values(
    [
        "cv_mean_bacc",
        "fold_wins_over_no_tta",
        "full_val_bacc",
        "cv_std_bacc",
    ],
    ascending=[
        False,
        False,
        False,
        True,
    ],
).reset_index(drop=True)

ROBUST_WEIGHT_DF = WEIGHT_SEARCH_DF[
    (
        WEIGHT_SEARCH_DF[
            "fold_wins_over_no_tta"
        ]
        >= MIN_FOLD_WINS
    )
    & (
        WEIGHT_SEARCH_DF[
            "full_val_gain_over_no_tta"
        ]
        > 0.0
    )
]

if len(ROBUST_WEIGHT_DF) > 0:
    WEIGHT_SELECTION_PASSED = True
    BEST_WEIGHT_ROW = ROBUST_WEIGHT_DF.iloc[0]
else:
    WEIGHT_SELECTION_PASSED = False
    BEST_WEIGHT_ROW = WEIGHT_SEARCH_DF.iloc[0]

BEST_WEIGHTS = np.asarray(
    [
        BEST_WEIGHT_ROW[
            f"weight__{model_name}"
        ]
        for model_name in MODEL_NAMES
    ],
    dtype=np.float32,
)
BEST_WEIGHTS /= BEST_WEIGHTS.sum()

WEIGHTED_VAL_PROB = (
    normalize_probabilities(
        np.tensordot(
            BEST_WEIGHTS,
            VAL_STACK,
            axes=(0, 0),
        )
    )
)

WEIGHTED_TEST_PROB = (
    normalize_probabilities(
        np.tensordot(
            BEST_WEIGHTS,
            TEST_STACK,
            axes=(0, 0),
        )
    )
)

WEIGHT_SEARCH_DF.to_csv(
    OUTPUT_DIR
    / "weighted_search_validation.csv",
    index=False,
)

print(
    "Weight candidates:",
    len(WEIGHT_SEARCH_DF),
)
print(
    "Robust candidates:",
    len(ROBUST_WEIGHT_DF),
)
print(
    "Robust selection passed:",
    WEIGHT_SELECTION_PASSED,
)

display(
    pd.DataFrame(
        {
            "model": MODEL_NAMES,
            "weight": BEST_WEIGHTS,
        }
    )
)


Weight candidates: 50072
Robust candidates: 35908
Robust selection passed: True


                model    weight
0        SimCLR_NoTTA  0.414650
1  SimCLR_SelectedTTA  0.090338
2            Rotation  0.324790
3        FusionConcat  0.104999
4       FusionGatedV2  0.002943
5       FusionGatedV3  0.039286
6      ResidualFusion  0.022994

In [7]:
# ============================================================
# CELL 07: VALIDATION-FOLD STACKING AND NO-TTA BLEND
# ============================================================

STACK_C_VALUES = [
    0.01,
    0.03,
    0.10,
    0.30,
    1.00,
    3.00,
    10.00,
]

STACK_ALPHA_VALUES = np.round(
    np.arange(
        0.05,
        0.66,
        0.05,
    ),
    2,
)


def entropy(probabilities):
    probabilities = np.clip(
        probabilities,
        1e-12,
        1.0,
    )

    return -np.sum(
        probabilities
        * np.log(probabilities),
        axis=1,
        keepdims=True,
    )


def build_meta_features(
    source_dictionary,
):
    blocks = []

    for model_name in MODEL_NAMES:
        probabilities = (
            source_dictionary[
                model_name
            ]
        )

        blocks.extend(
            [
                probabilities,
                np.log(
                    np.clip(
                        probabilities,
                        1e-12,
                        1.0,
                    )
                ),
                np.max(
                    probabilities,
                    axis=1,
                    keepdims=True,
                ),
                entropy(
                    probabilities
                ),
            ]
        )

    return np.concatenate(
        blocks,
        axis=1,
    ).astype(np.float32)


def make_meta_model(c_value):
    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=float(c_value),
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def align_probability(
    raw_probability,
    classes_seen,
):
    aligned = np.zeros(
        (
            raw_probability.shape[0],
            NUM_CLASSES,
        ),
        dtype=np.float32,
    )

    for source_column, class_id in enumerate(
        classes_seen
    ):
        aligned[
            :,
            int(class_id),
        ] = raw_probability[
            :,
            source_column,
        ]

    return normalize_probabilities(
        aligned
    )


META_X_VAL = build_meta_features(
    CALIBRATED_VAL
)
META_X_TEST = build_meta_features(
    CALIBRATED_TEST
)

stack_rows = []
stack_oof_probabilities = {}

for c_value in STACK_C_VALUES:
    oof_probability = np.zeros(
        (
            len(Y_VAL),
            NUM_CLASSES,
        ),
        dtype=np.float32,
    )

    for train_indices, fold_indices in FOLDS:
        model = make_meta_model(
            c_value
        )

        model.fit(
            META_X_VAL[
                train_indices
            ],
            Y_VAL[
                train_indices
            ],
        )

        raw_probability = model.predict_proba(
            META_X_VAL[
                fold_indices
            ]
        )

        classes_seen = (
            model.named_steps[
                "classifier"
            ].classes_
        )

        oof_probability[
            fold_indices
        ] = align_probability(
            raw_probability,
            classes_seen,
        )

    stack_oof_probabilities[
        c_value
    ] = oof_probability

    for alpha in STACK_ALPHA_VALUES:
        blended_probability = (
            normalize_probabilities(
                (
                    1.0 - alpha
                )
                * CALIBRATED_VAL[
                    ANCHOR_NAME
                ]
                + alpha
                * oof_probability
            )
        )

        blended_prediction = np.argmax(
            blended_probability,
            axis=1,
        )

        fold_bacc = np.asarray(
            [
                balanced_accuracy_score(
                    Y_VAL[
                        fold_indices
                    ],
                    blended_prediction[
                        fold_indices
                    ],
                )
                for _, fold_indices
                in FOLDS
            ]
        )

        full_bacc = (
            balanced_accuracy_score(
                Y_VAL,
                blended_prediction,
            )
        )

        stack_rows.append(
            {
                "C": float(c_value),
                "alpha": float(alpha),
                "cv_mean_bacc": float(
                    fold_bacc.mean()
                ),
                "cv_std_bacc": float(
                    fold_bacc.std()
                ),
                "full_val_bacc": float(
                    full_bacc
                ),
                "full_val_gain_over_no_tta": float(
                    full_bacc
                    - ANCHOR_FULL_BACC
                ),
                "fold_wins_over_no_tta": int(
                    np.sum(
                        fold_bacc
                        > (
                            ANCHOR_FOLD_BACC
                            + 1e-12
                        )
                    )
                ),
                "macro_f1": float(
                    f1_score(
                        Y_VAL,
                        blended_prediction,
                        average="macro",
                        zero_division=0,
                    )
                ),
            }
        )

STACK_SEARCH_DF = pd.DataFrame(
    stack_rows
).sort_values(
    [
        "cv_mean_bacc",
        "fold_wins_over_no_tta",
        "full_val_bacc",
        "macro_f1",
        "cv_std_bacc",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        True,
    ],
).reset_index(drop=True)

ROBUST_STACK_DF = STACK_SEARCH_DF[
    (
        STACK_SEARCH_DF[
            "fold_wins_over_no_tta"
        ]
        >= MIN_FOLD_WINS
    )
    & (
        STACK_SEARCH_DF[
            "full_val_gain_over_no_tta"
        ]
        > 0.0
    )
]

if len(ROBUST_STACK_DF) > 0:
    STACK_SELECTION_PASSED = True
    BEST_STACK_ROW = ROBUST_STACK_DF.iloc[0]
else:
    STACK_SELECTION_PASSED = False
    BEST_STACK_ROW = STACK_SEARCH_DF.iloc[0]

BEST_C = float(
    BEST_STACK_ROW["C"]
)
BEST_ALPHA = float(
    BEST_STACK_ROW["alpha"]
)

STACK_OOF_PROB = (
    stack_oof_probabilities[
        BEST_C
    ]
)

STACK_BLEND_VAL_PROB = (
    normalize_probabilities(
        (
            1.0 - BEST_ALPHA
        )
        * CALIBRATED_VAL[
            ANCHOR_NAME
        ]
        + BEST_ALPHA
        * STACK_OOF_PROB
    )
)

FINAL_META_MODEL = make_meta_model(
    BEST_C
)
FINAL_META_MODEL.fit(
    META_X_VAL,
    Y_VAL,
)

raw_test_probability = (
    FINAL_META_MODEL.predict_proba(
        META_X_TEST
    )
)

STACK_TEST_PROB = align_probability(
    raw_test_probability,
    FINAL_META_MODEL.named_steps[
        "classifier"
    ].classes_,
)

STACK_BLEND_TEST_PROB = (
    normalize_probabilities(
        (
            1.0 - BEST_ALPHA
        )
        * CALIBRATED_TEST[
            ANCHOR_NAME
        ]
        + BEST_ALPHA
        * STACK_TEST_PROB
    )
)

STACK_SEARCH_DF.to_csv(
    OUTPUT_DIR
    / "stacking_validation_search.csv",
    index=False,
)

joblib.dump(
    FINAL_META_MODEL,
    OUTPUT_DIR
    / "stacking_meta_model.joblib",
)

print(
    "Robust stacking candidates:",
    len(ROBUST_STACK_DF),
)
print(
    "Robust selection passed:",
    STACK_SELECTION_PASSED,
)
print(
    "Selected C:",
    BEST_C,
)
print(
    "Selected alpha:",
    BEST_ALPHA,
)
display(
    BEST_STACK_ROW.to_frame().T
)


Robust stacking candidates: 42
Robust selection passed: True
Selected C: 0.01
Selected alpha: 0.6


      C  alpha  cv_mean_bacc  cv_std_bacc  full_val_bacc  \
0  0.01    0.6      0.817028     0.024193       0.816158   

   full_val_gain_over_no_tta  fold_wins_over_no_tta  macro_f1  
0                   0.028312                    4.0  0.777824  

In [8]:
# ============================================================
# CELL 08: OPTIONAL VALIDATION-ONLY CLASS-BIAS CORRECTION
# ============================================================

BIAS_VALUES = [
    -0.15,
    -0.10,
    -0.05,
    0.00,
    0.05,
    0.10,
    0.15,
]


def apply_class_bias(
    probabilities,
    class_bias,
):
    logits = np.log(
        np.clip(
            normalize_probabilities(
                probabilities
            ),
            1e-12,
            1.0,
        )
    )

    logits += np.asarray(
        class_bias,
        dtype=np.float64,
    )[None, :]

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    return normalize_probabilities(
        np.exp(logits)
    )


def candidate_statistics(
    probabilities,
):
    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    fold_bacc = np.asarray(
        [
            balanced_accuracy_score(
                Y_VAL[
                    fold_indices
                ],
                predictions[
                    fold_indices
                ],
            )
            for _, fold_indices
            in FOLDS
        ]
    )

    return {
        "cv_mean_bacc": float(
            fold_bacc.mean()
        ),
        "cv_std_bacc": float(
            fold_bacc.std()
        ),
        "full_val_bacc": float(
            balanced_accuracy_score(
                Y_VAL,
                predictions,
            )
        ),
        "fold_wins_over_no_tta": int(
            np.sum(
                fold_bacc
                > (
                    ANCHOR_FOLD_BACC
                    + 1e-12
                )
            )
        ),
        "macro_f1": float(
            f1_score(
                Y_VAL,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
    }


CLASS_BIAS = np.zeros(
    NUM_CLASSES,
    dtype=np.float32,
)

for _ in range(2):
    for class_id in range(
        NUM_CLASSES
    ):
        best_choice = None

        for candidate_bias in BIAS_VALUES:
            trial_bias = CLASS_BIAS.copy()
            trial_bias[
                class_id
            ] = candidate_bias

            trial_probability = (
                apply_class_bias(
                    STACK_BLEND_VAL_PROB,
                    trial_bias,
                )
            )

            statistics = (
                candidate_statistics(
                    trial_probability
                )
            )

            score = (
                statistics[
                    "cv_mean_bacc"
                ]
                - 1e-4
                * np.sum(
                    np.abs(
                        trial_bias
                    )
                ),
                statistics[
                    "fold_wins_over_no_tta"
                ],
                statistics[
                    "full_val_bacc"
                ],
                statistics[
                    "macro_f1"
                ],
                -abs(candidate_bias),
            )

            if (
                best_choice is None
                or score
                > best_choice[0]
            ):
                best_choice = (
                    score,
                    candidate_bias,
                )

        CLASS_BIAS[
            class_id
        ] = best_choice[1]

STACK_BIAS_VAL_PROB = (
    apply_class_bias(
        STACK_BLEND_VAL_PROB,
        CLASS_BIAS,
    )
)

STACK_BIAS_TEST_PROB = (
    apply_class_bias(
        STACK_BLEND_TEST_PROB,
        CLASS_BIAS,
    )
)

print(
    "Selected class bias:",
    CLASS_BIAS.tolist(),
)
print(
    "No-bias statistics:",
    candidate_statistics(
        STACK_BLEND_VAL_PROB
    ),
)
print(
    "Class-bias statistics:",
    candidate_statistics(
        STACK_BIAS_VAL_PROB
    ),
)


Selected class bias: [-0.05000000074505806, 0.05000000074505806, -0.05000000074505806, 0.15000000596046448, 0.0, 0.0, 0.0, -0.05000000074505806]
No-bias statistics: {'cv_mean_bacc': 0.8170280963859851, 'cv_std_bacc': 0.02419277092398059, 'full_val_bacc': 0.8161584317359797, 'fold_wins_over_no_tta': 4, 'macro_f1': 0.7778236134123775}
Class-bias statistics: {'cv_mean_bacc': 0.821134232204763, 'cv_std_bacc': 0.02812556997537848, 'full_val_bacc': 0.8203316364511613, 'fold_wins_over_no_tta': 4, 'macro_f1': 0.7816339949384532}


In [9]:
# ============================================================
# CELL 09: FREEZE THE VALIDATION WINNER, THEN EVALUATE TEST
# ============================================================

VALIDATION_CANDIDATES = OrderedDict(
    [
        (
            "OptimizedWeightedEnsemble",
            (
                WEIGHTED_VAL_PROB,
                WEIGHTED_TEST_PROB,
            ),
        ),
        (
            "NoTTA_StackingBlend",
            (
                STACK_BLEND_VAL_PROB,
                STACK_BLEND_TEST_PROB,
            ),
        ),
        (
            "NoTTA_StackingBlend_ClassBias",
            (
                STACK_BIAS_VAL_PROB,
                STACK_BIAS_TEST_PROB,
            ),
        ),
    ]
)

candidate_rows = []

for method_name, (
    validation_probability,
    test_probability,
) in VALIDATION_CANDIDATES.items():
    statistics = candidate_statistics(
        validation_probability
    )

    candidate_rows.append(
        {
            "method": method_name,
            **statistics,
            "full_val_gain_over_no_tta": (
                statistics[
                    "full_val_bacc"
                ]
                - ANCHOR_FULL_BACC
            ),
        }
    )

CANDIDATE_DF = pd.DataFrame(
    candidate_rows
).sort_values(
    [
        "cv_mean_bacc",
        "fold_wins_over_no_tta",
        "full_val_bacc",
        "macro_f1",
        "cv_std_bacc",
    ],
    ascending=[
        False,
        False,
        False,
        False,
        True,
    ],
).reset_index(drop=True)

ROBUST_FINAL_DF = CANDIDATE_DF[
    (
        CANDIDATE_DF[
            "fold_wins_over_no_tta"
        ]
        >= MIN_FOLD_WINS
    )
    & (
        CANDIDATE_DF[
            "full_val_gain_over_no_tta"
        ]
        > 0.0
    )
]

if len(ROBUST_FINAL_DF) > 0:
    FINAL_SELECTION_PASSED = True
    FINAL_SELECTION_ROW = (
        ROBUST_FINAL_DF.iloc[0]
    )
else:
    FINAL_SELECTION_PASSED = False
    FINAL_SELECTION_ROW = (
        CANDIDATE_DF.iloc[0]
    )

FINAL_METHOD = str(
    FINAL_SELECTION_ROW["method"]
)

FINAL_VAL_PROB = (
    VALIDATION_CANDIDATES[
        FINAL_METHOD
    ][0]
)

FINAL_TEST_PROB = (
    VALIDATION_CANDIDATES[
        FINAL_METHOD
    ][1]
)

print("=" * 88)
print("VALIDATION-SELECTED FINAL ENSEMBLE")
print("=" * 88)
display(CANDIDATE_DF)
print(
    "\nFinal method:",
    FINAL_METHOD,
)
print(
    "Robust selection passed:",
    FINAL_SELECTION_PASSED,
)

# Test is evaluated only after the method above is frozen.
comparison_rows = []
result_objects = {}

for model_name in MODEL_NAMES:
    result = evaluate_probabilities(
        Y_TEST,
        TEST_PROBABILITIES[
            model_name
        ],
        model_name,
        "InternalTest",
        MODEL_CATEGORY.get(
            model_name,
            "Other",
        ),
    )

    result_objects[
        model_name
    ] = result

    comparison_rows.append(
        {
            key: value
            for key, value
            in result.items()
            if key not in {
                "y_pred",
                "confusion_matrix",
                "classification_report",
            }
        }
    )

for method_name, (
    validation_probability,
    test_probability,
) in VALIDATION_CANDIDATES.items():
    category = (
        "ProposedFinal"
        if method_name == FINAL_METHOD
        else "Ensemble"
    )

    result = evaluate_probabilities(
        Y_TEST,
        test_probability,
        method_name,
        "InternalTest",
        category,
    )

    result_objects[
        method_name
    ] = result

    comparison_rows.append(
        {
            key: value
            for key, value
            in result.items()
            if key not in {
                "y_pred",
                "confusion_matrix",
                "classification_report",
            }
        }
    )

# Reuse previous fixed weighted-ensemble predictions.
OLD_ENSEMBLE_VAL = (
    PIPELINE_ROOT
    / "simclr_dominant_weighted_ensemble_v2"
    / "WeightedEnsemble_val_predictions.csv"
)

OLD_ENSEMBLE_TEST = (
    PIPELINE_ROOT
    / "simclr_dominant_weighted_ensemble_v2"
    / "WeightedEnsemble_test_predictions.csv"
)

if (
    OLD_ENSEMBLE_VAL.exists()
    and OLD_ENSEMBLE_TEST.exists()
):
    old_y_test, old_test_prob = (
        load_prediction_csv(
            OLD_ENSEMBLE_TEST
        )
    )

    if np.array_equal(
        old_y_test,
        Y_TEST,
    ):
        old_result = (
            evaluate_probabilities(
                Y_TEST,
                old_test_prob,
                (
                    "PreviousFixed"
                    "WeightedEnsemble"
                ),
                "InternalTest",
                "Ensemble",
            )
        )

        result_objects[
            "PreviousFixedWeightedEnsemble"
        ] = old_result

        comparison_rows.append(
            {
                key: value
                for key, value
                in old_result.items()
                if key not in {
                    "y_pred",
                    "confusion_matrix",
                    "classification_report",
                }
            }
        )

COMPARISON_DF = pd.DataFrame(
    comparison_rows
).sort_values(
    [
        "bacc",
        "macro_f1",
        "accuracy",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).reset_index(drop=True)

COMPARISON_DF[
    "rank_by_test_bacc"
] = np.arange(
    1,
    len(COMPARISON_DF) + 1,
)

display(COMPARISON_DF)

final_result = result_objects[
    FINAL_METHOD
]
simclr_result = result_objects[
    "SimCLR_NoTTA"
]
rotation_result = result_objects[
    "Rotation"
]

single_bacc = [
    result["bacc"]
    for result in result_objects.values()
    if result.get(
        "category"
    ) == "Single"
]

fusion_bacc = [
    result["bacc"]
    for result in result_objects.values()
    if result.get(
        "category"
    ) == "Fusion"
]

BEAT_FLAGS = {
    "final_method": FINAL_METHOD,
    "final_test_bacc": float(
        final_result["bacc"]
    ),
    "simclr_no_tta_test_bacc": float(
        simclr_result["bacc"]
    ),
    "rotation_test_bacc": float(
        rotation_result["bacc"]
    ),
    "beats_simclr_no_tta": bool(
        final_result["bacc"]
        > simclr_result["bacc"]
    ),
    "beats_rotation": bool(
        final_result["bacc"]
        > rotation_result["bacc"]
    ),
    "beats_best_single_model": bool(
        final_result["bacc"]
        > max(single_bacc)
    ),
    "beats_every_fusion_model": bool(
        final_result["bacc"]
        > max(fusion_bacc)
    ),
}

print("\n" + "=" * 88)
print("EXPLICIT FINAL ANSWERS")
print("=" * 88)
print(
    json.dumps(
        BEAT_FLAGS,
        indent=2,
    )
)


VALIDATION-SELECTED FINAL ENSEMBLE


                          method  cv_mean_bacc  cv_std_bacc  full_val_bacc  \
0  NoTTA_StackingBlend_ClassBias      0.821134     0.028126       0.820332   
1      OptimizedWeightedEnsemble      0.818584     0.024204       0.817714   
2            NoTTA_StackingBlend      0.817028     0.024193       0.816158   

   fold_wins_over_no_tta  macro_f1  full_val_gain_over_no_tta  
0                      4  0.781634                   0.032485  
1                      5  0.804625                   0.029867  
2                      4  0.777824                   0.028312  


Final method: NoTTA_StackingBlend_ClassBias
Robust selection passed: True


                            model       category         split  n_samples  \
0             NoTTA_StackingBlend       Ensemble  InternalTest       2534   
1   NoTTA_StackingBlend_ClassBias  ProposedFinal  InternalTest       2534   
2                    SimCLR_NoTTA         Single  InternalTest       2534   
3       OptimizedWeightedEnsemble       Ensemble  InternalTest       2534   
4   PreviousFixedWeightedEnsemble       Ensemble  InternalTest       2534   
5              SimCLR_SelectedTTA         Single  InternalTest       2534   
6                  ResidualFusion         Fusion  InternalTest       2534   
7                    FusionConcat         Fusion  InternalTest       2534   
8                   FusionGatedV3         Fusion  InternalTest       2534   
9                   FusionGatedV2         Fusion  InternalTest       2534   
10                       Rotation         Single  InternalTest       2534   

    accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f


EXPLICIT FINAL ANSWERS
{
  "final_method": "NoTTA_StackingBlend_ClassBias",
  "final_test_bacc": 0.8395491150909984,
  "simclr_no_tta_test_bacc": 0.8107285352821415,
  "rotation_test_bacc": 0.7374427178422219,
  "beats_simclr_no_tta": true,
  "beats_rotation": true,
  "beats_best_single_model": true,
  "beats_every_fusion_model": true
}


In [10]:
# ============================================================
# CELL 10: SAVE FINAL INTERNAL RESULTS AND CONFIGURATION
# ============================================================

COMPARISON_DF.to_csv(
    OUTPUT_DIR
    / "internal_test_all_models_comparison.csv",
    index=False,
)

CANDIDATE_DF.to_csv(
    OUTPUT_DIR
    / "validation_final_ensemble_candidates.csv",
    index=False,
)

FINAL_CONFIG = {
    "pipeline_root": str(
        PIPELINE_ROOT
    ),
    "models": MODEL_NAMES,
    "anchor": ANCHOR_NAME,
    "temperatures": {
        model_name: float(
            TEMPERATURES[
                model_name
            ]
        )
        for model_name in MODEL_NAMES
    },
    "optimized_weights": {
        model_name: float(weight)
        for model_name, weight
        in zip(
            MODEL_NAMES,
            BEST_WEIGHTS,
        )
    },
    "stacking_C": BEST_C,
    "stacking_alpha": BEST_ALPHA,
    "class_bias": [
        float(value)
        for value in CLASS_BIAS
    ],
    "final_method": FINAL_METHOD,
    "validation_selection_passed": bool(
        FINAL_SELECTION_PASSED
    ),
    "validation_selection": (
        FINAL_SELECTION_ROW.to_dict()
    ),
    "test_used_for_selection": False,
    "test_beat_flags": BEAT_FLAGS,
}

with open(
    OUTPUT_DIR
    / "final_ensemble_config.json",
    "w",
) as file:
    json.dump(
        FINAL_CONFIG,
        file,
        indent=2,
        default=float,
    )

with open(
    OUTPUT_DIR
    / "internal_test_beat_flags.json",
    "w",
) as file:
    json.dump(
        BEAT_FLAGS,
        file,
        indent=2,
    )

np.savez_compressed(
    OUTPUT_DIR
    / "final_internal_probabilities.npz",
    y_val=Y_VAL,
    y_test=Y_TEST,
    final_val_prob=FINAL_VAL_PROB,
    final_test_prob=FINAL_TEST_PROB,
    weighted_val_prob=WEIGHTED_VAL_PROB,
    weighted_test_prob=WEIGHTED_TEST_PROB,
    stacking_val_prob=STACK_BLEND_VAL_PROB,
    stacking_test_prob=STACK_BLEND_TEST_PROB,
    stacking_bias_val_prob=STACK_BIAS_VAL_PROB,
    stacking_bias_test_prob=STACK_BIAS_TEST_PROB,
)

final_confusion = pd.DataFrame(
    final_result[
        "confusion_matrix"
    ],
    index=[
        f"true_{name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"pred_{name}"
        for name in CLASS_NAMES
    ],
)

final_confusion.to_csv(
    OUTPUT_DIR
    / "final_internal_confusion_matrix.csv"
)

pd.DataFrame(
    final_result[
        "classification_report"
    ]
).transpose().to_csv(
    OUTPUT_DIR
    / "final_internal_classification_report.csv"
)

print(
    "Saved final internal results to:",
    OUTPUT_DIR,
)


Saved final internal results to: /kaggle/working/reuse_only_all_model_ensemble


In [11]:
# ============================================================
# CELL 11: REUSE EXISTING SAVED ISIC/PAD RESULTS AS REFERENCES
# ============================================================
# This cell does not claim that the new ensemble was applied
# externally. The old notebook did not save each external base
# model probability array, so a new external ensemble cannot be
# reconstructed from summaries alone.

EXTERNAL_REFERENCE_FILES = {
    "Previous external summary": (
        PIPELINE_ROOT
        / "external_final_model_results"
        / "external_results_summary.csv"
    ),
    "PAD six-class zero-shot metrics": (
        PIPELINE_ROOT
        / "pad_ufes_20_six_class_restricted_evaluation"
        / "pad_ufes_20_six_class_metrics.csv"
    ),
    "PAD fine-tuned metrics": (
        PIPELINE_ROOT
        / "pad_ufes_20_six_class_finetuning"
        / "pad_six_class_test_metrics.csv"
    ),
}

external_reference_tables = []

for reference_name, reference_path in (
    EXTERNAL_REFERENCE_FILES.items()
):
    print(
        reference_name,
        ":",
        reference_path.exists(),
    )
    print(
        " ",
        reference_path,
    )

    if reference_path.exists():
        reference_df = pd.read_csv(
            reference_path
        )
        reference_df.insert(
            0,
            "reference_source",
            reference_name,
        )
        external_reference_tables.append(
            reference_df
        )

if external_reference_tables:
    EXTERNAL_REFERENCE_DF = pd.concat(
        external_reference_tables,
        ignore_index=True,
        sort=False,
    )

    display(
        EXTERNAL_REFERENCE_DF
    )

    EXTERNAL_REFERENCE_DF.to_csv(
        OUTPUT_DIR
        / "existing_isic_pad_results_reference.csv",
        index=False,
    )
else:
    print(
        "No previous external reference result files were found."
    )

print(
    "\nImportant: to apply the NEW all-model ensemble to ISIC/PAD, "
    "we need the per-model external probabilities. They were not "
    "saved by the previous notebook, so only a small inference-only "
    "notebook is needed later—not the full training pipeline."
)


Previous external summary : True
  /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/external_final_model_results/external_results_summary.csv
PAD six-class zero-shot metrics : True
  /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/pad_ufes_20_six_class_restricted_evaluation/pad_ufes_20_six_class_metrics.csv
PAD fine-tuned metrics : True
  /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike/pad_ufes_20_six_class_finetuning/pad_six_class_test_metrics.csv


                  reference_source                 dataset      final_method  \
0        Previous external summary  ISIC2019_Official_Test  WeightedEnsemble   
1        Previous external summary    PAD_UFES_20_External  WeightedEnsemble   
2  PAD six-class zero-shot metrics    PAD_UFES_20_External  WeightedEnsemble   
3           PAD fine-tuned metrics             PAD_UFES_20               NaN   

   n_samples  n_labeled                                    predictions_csv  \
0     8238.0     6191.0  /kaggle/working/rotation_ssl_paperlike/externa...   
1     2298.0     2298.0  /kaggle/working/rotation_ssl_paperlike/externa...   
2        NaN        NaN                                                NaN   
3      345.0        NaN                                                NaN   

                                     leaderboard_csv  accuracy      bacc  \
0  /kaggle/working/rotation_ssl_paperlike/externa...  0.688742  0.516902   
1  /kaggle/working/rotation_ssl_paperlike/externa...  0.


Important: to apply the NEW all-model ensemble to ISIC/PAD, we need the per-model external probabilities. They were not saved by the previous notebook, so only a small inference-only notebook is needed later—not the full training pipeline.


In [12]:
# ============================================================
# CELL 12: CREATE DOWNLOADABLE OUTPUT PACKAGE
# ============================================================

PACKAGE_BASE = Path(
    "/kaggle/working/"
    "reuse_only_all_model_ensemble_package"
)

ARCHIVE_PATH = shutil.make_archive(
    base_name=str(PACKAGE_BASE),
    format="zip",
    root_dir=str(OUTPUT_DIR),
)

print("=" * 88)
print("REUSE-ONLY ENSEMBLE COMPLETED")
print("=" * 88)
print(
    "Final method:",
    FINAL_METHOD,
)
print(
    "Final test BACC:",
    BEAT_FLAGS[
        "final_test_bacc"
    ],
)
print(
    "Beats no-TTA SimCLR:",
    BEAT_FLAGS[
        "beats_simclr_no_tta"
    ],
)
print(
    "Beats Rotation:",
    BEAT_FLAGS[
        "beats_rotation"
    ],
)
print(
    "Beats every fusion model:",
    BEAT_FLAGS[
        "beats_every_fusion_model"
    ],
)
print("\nZIP package:")
print(ARCHIVE_PATH)

print(
    "\nNow click Kaggle Save Version."
)


REUSE-ONLY ENSEMBLE COMPLETED
Final method: NoTTA_StackingBlend_ClassBias
Final test BACC: 0.8395491150909984
Beats no-TTA SimCLR: True
Beats Rotation: True
Beats every fusion model: True

ZIP package:
/kaggle/working/reuse_only_all_model_ensemble_package.zip

Now click Kaggle Save Version.


# External Inference Extension — Both Frozen Stacking Variants

This extension evaluates both frozen internal ensemble variants:

1. **NoTTA Stacking Blend** — no class-bias correction
2. **NoTTA Stacking Blend + Class Bias** — validation-selected proposed final model

The extension performs **inference only**. It does not retrain Rotation,
SimCLR, FusionConcat, FusionGated V2/V3, Residual Fusion, or the stacking
meta-classifier.

## Required Kaggle inputs

Attach:

- The saved output of `last-part-of-the-previous-one`
- ISIC 2019 official-test images and labels
- PAD-UFES-20 images and metadata

When this extension is run after the reuse-only internal notebook, the frozen
temperatures, model order, stacking meta-classifier, blend strength, and class
bias are reused directly.

The class-bias variant remains the proposed final model because it was selected
using internal validation. The no-bias variant is reported as a secondary
comparison; external test results are not used to redefine the final method.


In [13]:
# ============================================================
# EXTERNAL CELL E01: IMPORTS, FROZEN ENSEMBLE CHECKS, OUTPUT DIR
# ============================================================

import gc
import json
import os
import shutil
from collections import OrderedDict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50

EXTERNAL_OUTPUT_DIR = (
    Path("/kaggle/working")
    / "both_stacking_models_external_evaluation"
)
EXTERNAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXTERNAL_CACHE_DIR = (
    EXTERNAL_OUTPUT_DIR
    / "base_probability_cache"
)
EXTERNAL_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

IMG_SIZE = 224
EXTERNAL_BATCH_SIZE = 32

required_frozen_variables = [
    "PIPELINE_ROOT",
    "MODEL_NAMES",
    "TEMPERATURES",
    "FINAL_META_MODEL",
    "BEST_ALPHA",
    "CLASS_BIAS",
    "normalize_probabilities",
    "temperature_scale",
    "build_meta_features",
    "align_probability",
]

missing_frozen_variables = [
    name
    for name in required_frozen_variables
    if name not in globals()
]

if missing_frozen_variables:
    raise ValueError(
        "Run the reuse-only internal ensemble notebook first. "
        "These frozen variables are missing: "
        f"{missing_frozen_variables}"
    )

PIPELINE_ROOT = Path(PIPELINE_ROOT)

EXPECTED_EXTERNAL_MODELS = [
    "SimCLR_NoTTA",
    "SimCLR_SelectedTTA",
    "Rotation",
    "FusionConcat",
    "FusionGatedV2",
    "FusionGatedV3",
    "ResidualFusion",
]

if MODEL_NAMES != EXPECTED_EXTERNAL_MODELS:
    raise ValueError(
        "The frozen stacking model expects a different model order.\n"
        f"Expected by this extension: {EXPECTED_EXTERNAL_MODELS}\n"
        f"Frozen internal order: {MODEL_NAMES}"
    )

NO_BIAS_METHOD = "NoTTA_StackingBlend"
CLASS_BIAS_METHOD = "NoTTA_StackingBlend_ClassBias"

print("=" * 92)
print("EXTERNAL INFERENCE FOR BOTH FROZEN STACKING MODELS")
print("=" * 92)
print("Pipeline root:", PIPELINE_ROOT)
print("Frozen model order:", MODEL_NAMES)
print("Frozen alpha:", BEST_ALPHA)
print("Frozen class bias:", np.asarray(CLASS_BIAS).tolist())
print("Output directory:", EXTERNAL_OUTPUT_DIR)


EXTERNAL INFERENCE FOR BOTH FROZEN STACKING MODELS
Pipeline root: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike
Frozen model order: ['SimCLR_NoTTA', 'SimCLR_SelectedTTA', 'Rotation', 'FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']
Frozen alpha: 0.6
Frozen class bias: [-0.05000000074505806, 0.05000000074505806, -0.05000000074505806, 0.15000000596046448, 0.0, 0.0, 0.0, -0.05000000074505806]
Output directory: /kaggle/working/both_stacking_models_external_evaluation


In [14]:
# ============================================================
# EXTERNAL CELL E02: REMAP PREPARED ISIC/PAD TABLES TO INPUT IMAGES
# ============================================================

PREPARED_EXTERNAL_DIR = (
    PIPELINE_ROOT
    / "external_datasets_prepared"
)

ISIC_PREPARED_CSV = (
    PREPARED_EXTERNAL_DIR
    / "isic2019_official_test_prepared.csv"
)

PAD_PREPARED_CSV = (
    PREPARED_EXTERNAL_DIR
    / "pad_ufes_20_external_prepared.csv"
)

for required_path in [
    ISIC_PREPARED_CSV,
    PAD_PREPARED_CSV,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            "The previous notebook's prepared external table "
            f"was not found: {required_path}"
        )

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# Optional manual hints. Leave as None for automatic matching.
# Example:
# ISIC_PATH_HINT = "isic-2019-test"
# PAD_PATH_HINT = "pad-ufes-20"
ISIC_PATH_HINT = None
PAD_PATH_HINT = None


def normalized_image_key(value):
    value = str(value).strip()
    return Path(value).stem.lower()


def build_input_image_index():
    image_index = {}

    for path in Path("/kaggle/input").rglob("*"):
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            key = path.stem.lower()
            image_index.setdefault(
                key,
                [],
            ).append(path)

    return image_index


def choose_image_candidate(
    candidates,
    dataset_hint,
    manual_hint=None,
):
    if not candidates:
        return None

    ranked = []

    for candidate in candidates:
        text = str(candidate).lower()
        score = 0

        if manual_hint:
            if str(manual_hint).lower() in text:
                score += 100

        if dataset_hint == "isic":
            if "isic" in text:
                score += 30
            if "test" in text:
                score += 10
            if "pad" in text:
                score -= 50

        if dataset_hint == "pad":
            if "pad" in text:
                score += 30
            if "ufes" in text:
                score += 20
            if "isic" in text:
                score -= 50

        ranked.append(
            (
                -score,
                len(candidate.parts),
                str(candidate),
                candidate,
            )
        )

    ranked.sort()
    return ranked[0][-1]


def remap_prepared_dataframe(
    prepared_csv,
    dataset_hint,
    manual_hint=None,
):
    dataframe = pd.read_csv(
        prepared_csv
    )

    if "image_id" not in dataframe.columns:
        if "image_path" in dataframe.columns:
            dataframe["image_id"] = dataframe[
                "image_path"
            ].map(
                lambda value: Path(
                    str(value)
                ).stem
            )
        else:
            raise ValueError(
                f"{prepared_csv} has no image_id or image_path."
            )

    mapped_paths = []

    for image_id in dataframe["image_id"]:
        key = normalized_image_key(
            image_id
        )
        candidates = INPUT_IMAGE_INDEX.get(
            key,
            [],
        )
        selected = choose_image_candidate(
            candidates,
            dataset_hint=dataset_hint,
            manual_hint=manual_hint,
        )
        mapped_paths.append(
            str(selected)
            if selected is not None
            else None
        )

    dataframe = dataframe.copy()
    dataframe["image_path_current"] = mapped_paths
    dataframe["image_found"] = dataframe[
        "image_path_current"
    ].notna()

    missing_count = int(
        (~dataframe["image_found"]).sum()
    )

    print(
        f"{dataset_hint.upper()} prepared rows:",
        len(dataframe),
    )
    print(
        f"{dataset_hint.upper()} images mapped:",
        int(dataframe["image_found"].sum()),
    )
    print(
        f"{dataset_hint.upper()} images missing:",
        missing_count,
    )

    if missing_count > 0:
        display(
            dataframe.loc[
                ~dataframe["image_found"],
                ["image_id"],
            ].head(20)
        )

    dataframe = dataframe.loc[
        dataframe["image_found"]
    ].copy().reset_index(drop=True)

    if "label" not in dataframe.columns:
        raise ValueError(
            f"{prepared_csv} has no label column."
        )

    dataframe["label"] = dataframe[
        "label"
    ].fillna(-1).astype(int)

    return dataframe


print("Indexing attached input images...")
INPUT_IMAGE_INDEX = build_input_image_index()
print(
    "Unique image IDs indexed:",
    len(INPUT_IMAGE_INDEX),
)

ISIC_EXTERNAL_DF = remap_prepared_dataframe(
    ISIC_PREPARED_CSV,
    dataset_hint="isic",
    manual_hint=ISIC_PATH_HINT,
)

PAD_EXTERNAL_DF = remap_prepared_dataframe(
    PAD_PREPARED_CSV,
    dataset_hint="pad",
    manual_hint=PAD_PATH_HINT,
)

if len(ISIC_EXTERNAL_DF) == 0:
    raise RuntimeError(
        "No ISIC images were mapped. Set ISIC_PATH_HINT."
    )

if len(PAD_EXTERNAL_DF) == 0:
    raise RuntimeError(
        "No PAD images were mapped. Set PAD_PATH_HINT."
    )

print("\nISIC label distribution:")
print(
    ISIC_EXTERNAL_DF[
        "label"
    ].value_counts().sort_index()
)

print("\nPAD label distribution:")
print(
    PAD_EXTERNAL_DF[
        "label"
    ].value_counts().sort_index()
)


Indexing attached input images...
Unique image IDs indexed: 35877
ISIC prepared rows: 8238
ISIC images mapped: 8238
ISIC images missing: 0
PAD prepared rows: 2298
PAD images mapped: 2298
PAD images missing: 0

ISIC label distribution:
label
-1    2047
 0    1327
 1    2495
 2     975
 3     374
 4     660
 5      91
 6     104
 7     165
Name: count, dtype: int64

PAD label distribution:
label
0     52
1    244
2    845
3    730
4    235
7    192
Name: count, dtype: int64


In [15]:
# ============================================================
# EXTERNAL CELL E03: BUILD DETERMINISTIC IMAGE DATASETS
# ============================================================

def decode_external_image(
    image_path,
    label,
):
    image_bytes = tf.io.read_file(
        image_path
    )

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )

    image.set_shape(
        [None, None, 3]
    )

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        method="bilinear",
        antialias=True,
    )

    image = tf.cast(
        image,
        tf.float32,
    ) / 255.0

    return image, tf.cast(
        label,
        tf.int32,
    )


def build_external_dataset(
    dataframe,
    batch_size=EXTERNAL_BATCH_SIZE,
):
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            dataframe[
                "image_path_current"
            ].astype(str).values,
            dataframe[
                "label"
            ].astype(np.int32).values,
        )
    )

    dataset = dataset.map(
        decode_external_image,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False,
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


ISIC_EXTERNAL_DS = build_external_dataset(
    ISIC_EXTERNAL_DF
)

PAD_EXTERNAL_DS = build_external_dataset(
    PAD_EXTERNAL_DF
)

EXTERNAL_DATASETS_FOR_BOTH_STACKS = OrderedDict(
    [
        (
            "ISIC_2019_Official_Test",
            {
                "dataframe": ISIC_EXTERNAL_DF,
                "dataset": ISIC_EXTERNAL_DS,
            },
        ),
        (
            "PAD_UFES_20_External",
            {
                "dataframe": PAD_EXTERNAL_DF,
                "dataset": PAD_EXTERNAL_DS,
            },
        ),
    ]
)

for dataset_name, payload in (
    EXTERNAL_DATASETS_FOR_BOTH_STACKS.items()
):
    first_images, first_labels = next(
        iter(payload["dataset"])
    )
    print(
        dataset_name,
        "batch:",
        first_images.shape,
        first_labels.shape,
    )


I0000 00:00:1784050713.920410      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


ISIC_2019_Official_Test batch: (32, 224, 224, 3) (32,)
PAD_UFES_20_External batch: (32, 224, 224, 3) (32,)


In [16]:
# ============================================================
# EXTERNAL CELL E04: INFERENCE-ONLY MODEL ARCHITECTURES
# ============================================================

SIMCLR_DROPOUT = 0.30
ROTATION_FT_DROPOUT = 0.30

FUSIONCONCAT_DROPOUT = 0.40
FUSIONCONCAT_HIDDEN_DIM_1 = 512
FUSIONCONCAT_HIDDEN_DIM_2 = 256
FUSIONCONCAT_WEIGHT_DECAY = 1e-4

FUSIONGATED_V2_DROPOUT = 0.50
FUSIONGATED_V2_WEIGHT_DECAY = 3e-4
FUSIONGATED_V2_BRANCH_DIM = 256
FUSIONGATED_V2_GATE_HIDDEN = 128
FUSIONGATED_V2_HEAD_HIDDEN = 128
FUSIONGATED_V2_TEMPERATURE = 1.5

BESTV3_BRANCH_DIM = 256
BESTV3_DROPOUT_1 = 0.50
BESTV3_DROPOUT_2 = 0.40
BESTV3_WEIGHT_DECAY = 4e-4
BESTV3_GATE_HIDDEN = 128
BESTV3_HEAD_HIDDEN_1 = 256
BESTV3_HEAD_HIDDEN_2 = 128
BESTV3_TEMPERATURE = 1.0

RESIDUAL_FUSION_BRANCH_DIM = 256
RESIDUAL_FUSION_HIDDEN_DIM = 256
RESIDUAL_FUSION_WEIGHT_DECAY = 3e-4
RESIDUAL_FUSION_DROPOUT = 0.40
RESIDUAL_FUSION_MAX_ALPHA = 0.35


def resnet_preprocess_tensor(images):
    images = tf.cast(
        images,
        tf.float32,
    ) * 255.0

    return tf.keras.applications.resnet50.preprocess_input(
        images
    )


def build_resnet50_encoder(
    name,
    training_flag=False,
):
    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name=f"{name}_input",
    )

    backbone = ResNet50(
        include_top=False,
        weights=None,
        input_shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        pooling="avg",
    )

    features = backbone(
        inputs,
        training=training_flag,
    )

    return keras.Model(
        inputs,
        features,
        name=name,
    )


def build_simclr_finetune_model():
    encoder = build_resnet50_encoder(
        "simclr_resnet50_encoder",
        training_flag=False,
    )

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="finetune_img",
    )

    x = layers.Lambda(
        resnet_preprocess_tensor,
        name="resnet_preprocess",
    )(inputs)

    x = encoder(
        x,
        training=False,
    )

    x = layers.Dropout(
        SIMCLR_DROPOUT,
        name="finetune_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="finetune_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="simclr_finetune_model",
    )

    return model, encoder


def build_rotation_finetune_model():
    encoder = build_resnet50_encoder(
        "rotation_ft_resnet50_encoder",
        training_flag=True,
    )
    encoder.trainable = True

    inputs = keras.Input(
        shape=(
            IMG_SIZE,
            IMG_SIZE,
            3,
        ),
        name="imagenet_rotation_input",
    )

    x = layers.Lambda(
        resnet_preprocess_tensor,
        name="resnet50_preprocess",
    )(inputs)

    # This reproduces the saved architecture.
    x = encoder(
        x,
        training=True
        if encoder.trainable
        else False,
    )

    x = layers.Dropout(
        ROTATION_FT_DROPOUT,
        name="imagenet_rotation_dropout",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="imagenet_rotation_classifier",
    )(x)

    model = keras.Model(
        inputs,
        outputs,
        name="imagenet_rotation_finetune_model",
    )

    return model


def l2_normalize_features(
    features,
    epsilon=1e-12,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    denominator = np.linalg.norm(
        features,
        axis=1,
        keepdims=True,
    )

    return (
        features
        / np.clip(
            denominator,
            epsilon,
            None,
        )
    ).astype(np.float32)


def build_fusionconcat_model(
    input_dim,
):
    inputs = keras.Input(
        shape=(input_dim,),
        name="fusionconcat_input",
    )

    x = layers.BatchNormalization(
        name="bn_input"
    )(inputs)

    x = layers.Dense(
        FUSIONCONCAT_HIDDEN_DIM_1,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONCONCAT_WEIGHT_DECAY
            )
        ),
        name="fusionconcat_dense_1",
    )(x)

    x = layers.BatchNormalization(
        name="bn_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_1",
    )(x)
    x = layers.Dropout(
        FUSIONCONCAT_DROPOUT,
        name="dropout_1",
    )(x)

    x = layers.Dense(
        FUSIONCONCAT_HIDDEN_DIM_2,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONCONCAT_WEIGHT_DECAY
            )
        ),
        name="fusionconcat_dense_2",
    )(x)

    x = layers.BatchNormalization(
        name="bn_2"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_2",
    )(x)
    x = layers.Dropout(
        FUSIONCONCAT_DROPOUT,
        name="dropout_2",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="fusionconcat_classifier",
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name="fusionconcat_model",
    )


def stable_softmax_temperature(
    values,
    temperature,
):
    return tf.nn.softmax(
        values / temperature,
        axis=-1,
    )


def build_fusiongated_v2_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_input",
    )

    simclr_norm = layers.LayerNormalization(
        name="ln_simclr_in"
    )(simclr_input)

    rotation_norm = layers.LayerNormalization(
        name="ln_rotation_in"
    )(rotation_input)

    simclr_projection = layers.Dense(
        FUSIONGATED_V2_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="simclr_proj",
    )(simclr_norm)

    simclr_projection = layers.LayerNormalization(
        name="ln_simclr_proj"
    )(simclr_projection)
    simclr_projection = layers.Activation(
        "relu",
        name="relu_simclr_proj",
    )(simclr_projection)

    rotation_projection = layers.Dense(
        FUSIONGATED_V2_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="rotation_proj",
    )(rotation_norm)

    rotation_projection = layers.LayerNormalization(
        name="ln_rotation_proj"
    )(rotation_projection)
    rotation_projection = layers.Activation(
        "relu",
        name="relu_rotation_proj",
    )(rotation_projection)

    gate_input = layers.Concatenate(
        name="gate_input_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    gate_hidden = layers.Dense(
        FUSIONGATED_V2_GATE_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="gate_dense_1",
    )(gate_input)

    gate_hidden = layers.LayerNormalization(
        name="ln_gate_1"
    )(gate_hidden)
    gate_hidden = layers.Activation(
        "relu",
        name="relu_gate_1",
    )(gate_hidden)
    gate_hidden = layers.Dropout(
        FUSIONGATED_V2_DROPOUT,
        name="dropout_gate_1",
    )(gate_hidden)

    gate_logits = layers.Dense(
        2,
        activation=None,
        bias_initializer=(
            keras.initializers.Constant(
                [0.30, 0.00]
            )
        ),
        name="gate_logits",
    )(gate_hidden)

    gate_weights = layers.Lambda(
        lambda values: (
            stable_softmax_temperature(
                values,
                FUSIONGATED_V2_TEMPERATURE,
            )
        ),
        name="branch_gate_softmax",
    )(gate_logits)

    alpha = layers.Lambda(
        lambda values: values[:, 0:1],
        name="alpha_weight",
    )(gate_weights)

    beta = layers.Lambda(
        lambda values: values[:, 1:2],
        name="beta_weight",
    )(gate_weights)

    weighted_simclr = layers.Multiply(
        name="weighted_simclr"
    )(
        [
            simclr_projection,
            alpha,
        ]
    )

    weighted_rotation = layers.Multiply(
        name="weighted_rotation"
    )(
        [
            rotation_projection,
            beta,
        ]
    )

    weighted_sum = layers.Add(
        name="weighted_sum"
    )(
        [
            weighted_simclr,
            weighted_rotation,
        ]
    )

    fused = layers.Concatenate(
        name="residual_fused_concat"
    )(
        [
            weighted_sum,
            simclr_projection,
            rotation_projection,
        ]
    )

    x = layers.LayerNormalization(
        name="ln_fused"
    )(fused)

    x = layers.Dense(
        FUSIONGATED_V2_HEAD_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                FUSIONGATED_V2_WEIGHT_DECAY
            )
        ),
        name="head_dense_1",
    )(x)

    x = layers.LayerNormalization(
        name="ln_head_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_1",
    )(x)
    x = layers.Dropout(
        FUSIONGATED_V2_DROPOUT,
        name="dropout_head_1",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(x)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
        ],
        outputs,
        name="fusiongated_v2_model",
    )


def build_fusiongated_v3_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_input",
    )

    simclr_norm = layers.LayerNormalization(
        name="ln_simclr_in"
    )(simclr_input)
    rotation_norm = layers.LayerNormalization(
        name="ln_rotation_in"
    )(rotation_input)

    simclr_projection = layers.Dense(
        BESTV3_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="simclr_proj",
    )(simclr_norm)
    simclr_projection = layers.LayerNormalization(
        name="ln_simclr_proj"
    )(simclr_projection)
    simclr_projection = layers.Activation(
        "relu",
        name="relu_simclr_proj",
    )(simclr_projection)

    rotation_projection = layers.Dense(
        BESTV3_BRANCH_DIM,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="rotation_proj",
    )(rotation_norm)
    rotation_projection = layers.LayerNormalization(
        name="ln_rotation_proj"
    )(rotation_projection)
    rotation_projection = layers.Activation(
        "relu",
        name="relu_rotation_proj",
    )(rotation_projection)

    gate_input = layers.Concatenate(
        name="gate_input_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    gate_hidden = layers.Dense(
        BESTV3_GATE_HIDDEN,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="gate_dense_1",
    )(gate_input)
    gate_hidden = layers.LayerNormalization(
        name="ln_gate_1"
    )(gate_hidden)
    gate_hidden = layers.Activation(
        "relu",
        name="relu_gate_1",
    )(gate_hidden)
    gate_hidden = layers.Dropout(
        BESTV3_DROPOUT_1,
        name="dropout_gate_1",
    )(gate_hidden)

    gate_logits = layers.Dense(
        2,
        activation=None,
        bias_initializer=(
            keras.initializers.Constant(
                [0.35, 0.00]
            )
        ),
        name="gate_logits",
    )(gate_hidden)

    gate_weights = layers.Lambda(
        lambda values: (
            stable_softmax_temperature(
                values,
                BESTV3_TEMPERATURE,
            )
        ),
        name="branch_gate_softmax",
    )(gate_logits)

    alpha = layers.Lambda(
        lambda values: values[:, 0:1],
        name="alpha_weight",
    )(gate_weights)
    beta = layers.Lambda(
        lambda values: values[:, 1:2],
        name="beta_weight",
    )(gate_weights)

    weighted_simclr = layers.Multiply(
        name="weighted_simclr"
    )(
        [
            simclr_projection,
            alpha,
        ]
    )
    weighted_rotation = layers.Multiply(
        name="weighted_rotation"
    )(
        [
            rotation_projection,
            beta,
        ]
    )
    weighted_sum = layers.Add(
        name="weighted_sum"
    )(
        [
            weighted_simclr,
            weighted_rotation,
        ]
    )

    absolute_difference = layers.Lambda(
        lambda values: tf.abs(
            values[0] - values[1]
        ),
        name="abs_diff",
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    elementwise_product = layers.Multiply(
        name="elem_prod"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    fused = layers.Concatenate(
        name="final_fused_concat"
    )(
        [
            weighted_sum,
            simclr_projection,
            rotation_projection,
            absolute_difference,
            elementwise_product,
        ]
    )

    x = layers.LayerNormalization(
        name="ln_fused"
    )(fused)

    x = layers.Dense(
        BESTV3_HEAD_HIDDEN_1,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="head_dense_1",
    )(x)
    x = layers.LayerNormalization(
        name="ln_head_1"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_1",
    )(x)
    x = layers.Dropout(
        BESTV3_DROPOUT_1,
        name="dropout_head_1",
    )(x)

    x = layers.Dense(
        BESTV3_HEAD_HIDDEN_2,
        activation=None,
        kernel_regularizer=(
            keras.regularizers.l2(
                BESTV3_WEIGHT_DECAY
            )
        ),
        name="head_dense_2",
    )(x)
    x = layers.LayerNormalization(
        name="ln_head_2"
    )(x)
    x = layers.Activation(
        "relu",
        name="relu_head_2",
    )(x)
    x = layers.Dropout(
        BESTV3_DROPOUT_2,
        name="dropout_head_2",
    )(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(x)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
        ],
        outputs,
        name="fusiongated_best_v3_model",
    )


class BoundedResidualLogitFusion(
    layers.Layer
):
    def __init__(
        self,
        max_alpha=0.35,
        initial_alpha=0.08,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.max_alpha = float(
            max_alpha
        )
        ratio = np.clip(
            initial_alpha / max_alpha,
            1e-4,
            1.0 - 1e-4,
        )
        self.initial_logit = float(
            np.log(
                ratio
                / (1.0 - ratio)
            )
        )

    def build(
        self,
        input_shape,
    ):
        self.alpha_logit = (
            self.add_weight(
                name="alpha_logit",
                shape=(),
                initializer=(
                    keras.initializers.Constant(
                        self.initial_logit
                    )
                ),
                trainable=True,
            )
        )
        super().build(
            input_shape
        )

    def call(
        self,
        inputs,
    ):
        base_logits, correction_logits = inputs
        alpha = (
            self.max_alpha
            * tf.sigmoid(
                self.alpha_logit
            )
        )
        return (
            base_logits
            + alpha
            * correction_logits
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_alpha": self.max_alpha,
            }
        )
        return config


def build_residual_fusion_model(
    simclr_dim,
    rotation_dim,
):
    simclr_input = keras.Input(
        shape=(simclr_dim,),
        name="simclr_feature_input",
    )
    rotation_input = keras.Input(
        shape=(rotation_dim,),
        name="rotation_feature_input",
    )
    base_logits_input = keras.Input(
        shape=(NUM_CLASSES,),
        name="base_simclr_logits",
    )

    simclr_projection = (
        layers.LayerNormalization(
            name="sim_ln_in"
        )(simclr_input)
    )
    simclr_projection = layers.Dense(
        RESIDUAL_FUSION_BRANCH_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="sim_projection",
    )(simclr_projection)
    simclr_projection = (
        layers.LayerNormalization(
            name="sim_ln_proj"
        )(simclr_projection)
    )
    simclr_projection = layers.Activation(
        "relu",
        name="sim_relu",
    )(simclr_projection)

    rotation_projection = (
        layers.LayerNormalization(
            name="rot_ln_in"
        )(rotation_input)
    )
    rotation_projection = layers.Dense(
        RESIDUAL_FUSION_BRANCH_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="rot_projection",
    )(rotation_projection)
    rotation_projection = (
        layers.LayerNormalization(
            name="rot_ln_proj"
        )(rotation_projection)
    )
    rotation_projection = layers.Activation(
        "relu",
        name="rot_relu",
    )(rotation_projection)

    absolute_difference = layers.Lambda(
        lambda values: tf.abs(
            values[0] - values[1]
        ),
        name="abs_difference",
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    product = layers.Multiply(
        name="elementwise_product"
    )(
        [
            simclr_projection,
            rotation_projection,
        ]
    )

    fused = layers.Concatenate(
        name="residual_feature_concat"
    )(
        [
            simclr_projection,
            rotation_projection,
            absolute_difference,
            product,
        ]
    )
    fused = layers.LayerNormalization(
        name="fusion_ln"
    )(fused)
    fused = layers.Dense(
        RESIDUAL_FUSION_HIDDEN_DIM,
        kernel_regularizer=(
            keras.regularizers.l2(
                RESIDUAL_FUSION_WEIGHT_DECAY
            )
        ),
        name="correction_dense",
    )(fused)
    fused = layers.LayerNormalization(
        name="correction_ln"
    )(fused)
    fused = layers.Activation(
        "relu",
        name="correction_relu",
    )(fused)
    fused = layers.Dropout(
        RESIDUAL_FUSION_DROPOUT,
        name="correction_dropout",
    )(fused)
    correction_logits = layers.Dense(
        NUM_CLASSES,
        name="correction_logits",
    )(fused)

    final_logits = (
        BoundedResidualLogitFusion(
            max_alpha=(
                RESIDUAL_FUSION_MAX_ALPHA
            ),
            initial_alpha=0.08,
            name=(
                "bounded_residual_"
                "logit_fusion"
            ),
        )(
            [
                base_logits_input,
                correction_logits,
            ]
        )
    )

    output = layers.Softmax(
        dtype="float32",
        name="final_probabilities",
    )(final_logits)

    return keras.Model(
        [
            simclr_input,
            rotation_input,
            base_logits_input,
        ],
        output,
        name=(
            "simclr_preserving_"
            "residual_fusion"
        ),
    )


In [17]:
# ============================================================
# EXTERNAL CELL E05: DISCOVER AND VERIFY SAVED MODEL WEIGHTS
# ============================================================

SIMCLR_DIR = (
    PIPELINE_ROOT
    / "simclr_finetune_multiseed_bacc_v2"
)

SIMCLR_CONFIG_JSON = (
    SIMCLR_DIR
    / "multiseed_config.json"
)

SIMCLR_RESULTS_CSV = (
    SIMCLR_DIR
    / "multiseed_results.csv"
)

SIMCLR_TTA_CONFIG_JSON = (
    SIMCLR_DIR
    / "tta_selection"
    / "selected_tta_policy.json"
)

with open(
    SIMCLR_CONFIG_JSON,
    "r",
) as file:
    simclr_config = json.load(file)

SIMCLR_SEEDS = [
    int(seed)
    for seed in simclr_config[
        "seeds"
    ]
]

SIMCLR_SEED_PATHS = OrderedDict(
    [
        (
            seed,
            SIMCLR_DIR
            / f"seed_{seed}"
            / "best_by_val_bacc.weights.h5",
        )
        for seed in SIMCLR_SEEDS
    ]
)

simclr_results_df = pd.read_csv(
    SIMCLR_RESULTS_CSV
)

representative_row = (
    simclr_results_df.sort_values(
        [
            "val_bacc",
            "val_macro_f1",
            "val_accuracy",
            "seed",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    ).iloc[0]
)

SIMCLR_REPRESENTATIVE_SEED = int(
    representative_row["seed"]
)

with open(
    SIMCLR_TTA_CONFIG_JSON,
    "r",
) as file:
    selected_tta_config = json.load(
        file
    )

SELECTED_TTA_TRANSFORMS = list(
    selected_tta_config[
        "transforms"
    ]
)

ROTATION_SSL_ENCODER_PATH = (
    PIPELINE_ROOT
    / "rotation_ssl"
    / "rotation_ssl_encoder_best.weights.h5"
)

ROTATION_FT_PATH = (
    PIPELINE_ROOT
    / "extended_imagenet_rotation_finetune"
    / "imagenet_rotation_finetune_best.weights.h5"
)

FUSIONCONCAT_PATH = (
    PIPELINE_ROOT
    / "fusion_concat_multiseed_rep_v2"
    / "fusionconcat_best.weights.h5"
)

FUSIONGATED_V2_PATH = (
    PIPELINE_ROOT
    / "fusion_gated_v2_multiseed_rep_v2"
    / "fusiongated_v2_best.weights.h5"
)

FUSIONGATED_V3_PATH = (
    PIPELINE_ROOT
    / "fusion_gated_v3_multiseed_rep_v2"
    / "fusiongated_best_v3.weights.h5"
)

RESIDUAL_DIR = (
    PIPELINE_ROOT
    / "residual_simclr_rotation_fusion_v1"
)

RESIDUAL_CONFIG_JSON = (
    RESIDUAL_DIR
    / "config.json"
)

with open(
    RESIDUAL_CONFIG_JSON,
    "r",
) as file:
    residual_config = json.load(
        file
    )

RESIDUAL_SEEDS = [
    int(seed)
    for seed in residual_config[
        "seeds"
    ]
]

RESIDUAL_SEED_PATHS = OrderedDict(
    [
        (
            seed,
            RESIDUAL_DIR
            / f"seed_{seed}"
            / "best_by_val_bacc.weights.h5",
        )
        for seed in RESIDUAL_SEEDS
    ]
)

artifact_rows = []

for name, path in {
    **{
        f"SimCLR seed {seed}": path
        for seed, path
        in SIMCLR_SEED_PATHS.items()
    },
    "Rotation SSL encoder": (
        ROTATION_SSL_ENCODER_PATH
    ),
    "Standalone Rotation": (
        ROTATION_FT_PATH
    ),
    "FusionConcat": (
        FUSIONCONCAT_PATH
    ),
    "FusionGated V2": (
        FUSIONGATED_V2_PATH
    ),
    "FusionGated V3": (
        FUSIONGATED_V3_PATH
    ),
    **{
        f"Residual Fusion seed {seed}": path
        for seed, path
        in RESIDUAL_SEED_PATHS.items()
    },
}.items():
    artifact_rows.append(
        {
            "artifact": name,
            "path": str(path),
            "exists": path.exists(),
        }
    )

ARTIFACT_DF = pd.DataFrame(
    artifact_rows
)

display(ARTIFACT_DF)

missing_artifacts = ARTIFACT_DF.loc[
    ~ARTIFACT_DF["exists"]
]

if len(missing_artifacts) > 0:
    raise FileNotFoundError(
        "Some saved inference artifacts are missing:\n"
        + missing_artifacts.to_string(
            index=False
        )
    )

print(
    "Representative SimCLR seed:",
    SIMCLR_REPRESENTATIVE_SEED,
)
print(
    "Selected TTA transforms:",
    SELECTED_TTA_TRANSFORMS,
)
print(
    "Residual seeds:",
    RESIDUAL_SEEDS,
)


                   artifact  \
0            SimCLR seed 21   
1            SimCLR seed 35   
2            SimCLR seed 42   
3            SimCLR seed 52   
4            SimCLR seed 62   
5      Rotation SSL encoder   
6       Standalone Rotation   
7              FusionConcat   
8            FusionGated V2   
9            FusionGated V3   
10  Residual Fusion seed 21   
11  Residual Fusion seed 42   
12  Residual Fusion seed 62   

                                                 path  exists  
0   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
1   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
2   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
3   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
4   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
5   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
6   /kaggle/input/notebooks/anirbanroysudipto/last...    True  
7   /kaggle/input/notebooks/anirbanroysudipto/last...

Representative SimCLR seed: 62
Selected TTA transforms: ['identity', 'hflip']
Residual seeds: [21, 42, 62]


In [18]:
# ============================================================
# EXTERNAL CELL E06: BASE-MODEL EXTERNAL INFERENCE WITH CACHING
# ============================================================

def apply_tta_transform(
    images,
    transform_name,
):
    if transform_name == "identity":
        return images
    if transform_name == "hflip":
        return tf.image.flip_left_right(
            images
        )
    if transform_name == "vflip":
        return tf.image.flip_up_down(
            images
        )
    if transform_name == "rot90":
        return tf.image.rot90(
            images,
            k=1,
        )
    if transform_name == "rot180":
        return tf.image.rot90(
            images,
            k=2,
        )
    if transform_name == "rot270":
        return tf.image.rot90(
            images,
            k=3,
        )

    raise ValueError(
        f"Unknown TTA transform: {transform_name}"
    )


def predict_simclr_seed_both(
    model,
    dataset,
    tta_transforms,
):
    no_tta_batches = []
    selected_tta_batches = []
    labels = []

    for images, batch_labels in dataset:
        identity_probability = np.asarray(
            model(
                images,
                training=False,
            ),
            dtype=np.float32,
        )

        view_probabilities = []

        for transform_name in tta_transforms:
            if transform_name == "identity":
                transformed_probability = (
                    identity_probability
                )
            else:
                transformed_images = (
                    apply_tta_transform(
                        images,
                        transform_name,
                    )
                )
                transformed_probability = (
                    np.asarray(
                        model(
                            transformed_images,
                            training=False,
                        ),
                        dtype=np.float32,
                    )
                )

            view_probabilities.append(
                transformed_probability
            )

        selected_probability = np.mean(
            np.stack(
                view_probabilities,
                axis=0,
            ),
            axis=0,
        )

        no_tta_batches.append(
            identity_probability
        )
        selected_tta_batches.append(
            selected_probability
        )
        labels.append(
            np.asarray(
                batch_labels,
                dtype=np.int32,
            )
        )

    return (
        normalize_probabilities(
            np.concatenate(
                no_tta_batches,
                axis=0,
            )
        ),
        normalize_probabilities(
            np.concatenate(
                selected_tta_batches,
                axis=0,
            )
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def extract_encoder_features(
    encoder,
    dataset,
):
    feature_batches = []
    labels = []

    for images, batch_labels in dataset:
        preprocessed = (
            resnet_preprocess_tensor(
                images
            )
        )

        features = encoder(
            preprocessed,
            training=False,
        )

        feature_batches.append(
            np.asarray(
                features,
                dtype=np.float32,
            )
        )

        labels.append(
            np.asarray(
                batch_labels,
                dtype=np.int32,
            )
        )

    return (
        l2_normalize_features(
            np.concatenate(
                feature_batches,
                axis=0,
            )
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def predict_direct_model(
    model,
    dataset,
):
    probability_batches = []
    labels = []

    for images, batch_labels in dataset:
        probability_batches.append(
            np.asarray(
                model(
                    images,
                    training=False,
                ),
                dtype=np.float32,
            )
        )
        labels.append(
            np.asarray(
                batch_labels,
                dtype=np.int32,
            )
        )

    return (
        normalize_probabilities(
            np.concatenate(
                probability_batches,
                axis=0,
            )
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )


def generate_external_base_probabilities(
    dataset_name,
    dataset,
    expected_labels,
):
    cache_path = (
        EXTERNAL_CACHE_DIR
        / f"{dataset_name}_all_base_probabilities.npz"
    )

    if cache_path.exists():
        with np.load(
            cache_path
        ) as saved:
            cached_names = [
                str(value)
                for value in saved[
                    "model_names"
                ].tolist()
            ]

            if cached_names == MODEL_NAMES:
                cached_labels = np.asarray(
                    saved["labels"],
                    dtype=np.int32,
                )

                if np.array_equal(
                    cached_labels,
                    expected_labels,
                ):
                    print(
                        "Using cached external base probabilities:",
                        cache_path,
                    )

                    return OrderedDict(
                        [
                            (
                                model_name,
                                normalize_probabilities(
                                    saved[
                                        f"prob__{model_name}"
                                    ]
                                ),
                            )
                            for model_name
                            in MODEL_NAMES
                        ]
                    )

    print(
        "Generating external base probabilities for",
        dataset_name,
    )

    # --------------------------------------------------------
    # 1. Multi-seed SimCLR: no-TTA and selected TTA.
    # --------------------------------------------------------
    no_tta_seed_probabilities = []
    selected_tta_seed_probabilities = []
    representative_probability = None

    for seed, weight_path in (
        SIMCLR_SEED_PATHS.items()
    ):
        print(
            f"SimCLR external inference, seed {seed}"
        )

        tf.keras.backend.clear_session()
        gc.collect()

        model, _ = (
            build_simclr_finetune_model()
        )
        model.load_weights(
            str(weight_path)
        )

        (
            seed_no_tta,
            seed_selected_tta,
            labels_check,
        ) = predict_simclr_seed_both(
            model,
            dataset,
            SELECTED_TTA_TRANSFORMS,
        )

        if not np.array_equal(
            labels_check,
            expected_labels,
        ):
            raise ValueError(
                "External label order changed during "
                f"SimCLR seed {seed} inference."
            )

        no_tta_seed_probabilities.append(
            seed_no_tta
        )
        selected_tta_seed_probabilities.append(
            seed_selected_tta
        )

        if (
            seed
            == SIMCLR_REPRESENTATIVE_SEED
        ):
            representative_probability = (
                seed_no_tta
            )

        del model
        gc.collect()

    simclr_no_tta_probability = (
        normalize_probabilities(
            np.mean(
                np.stack(
                    no_tta_seed_probabilities,
                    axis=0,
                ),
                axis=0,
            )
        )
    )

    simclr_selected_tta_probability = (
        normalize_probabilities(
            np.mean(
                np.stack(
                    selected_tta_seed_probabilities,
                    axis=0,
                ),
                axis=0,
            )
        )
    )

    if representative_probability is None:
        raise RuntimeError(
            "The representative SimCLR seed probability "
            "was not generated."
        )

    # --------------------------------------------------------
    # 2. Extract representative SimCLR and Rotation features.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    representative_model, representative_encoder = (
        build_simclr_finetune_model()
    )
    representative_model.load_weights(
        str(
            SIMCLR_SEED_PATHS[
                SIMCLR_REPRESENTATIVE_SEED
            ]
        )
    )

    (
        simclr_features,
        feature_labels,
    ) = extract_encoder_features(
        representative_encoder,
        dataset,
    )

    del representative_model
    del representative_encoder
    gc.collect()

    if not np.array_equal(
        feature_labels,
        expected_labels,
    ):
        raise ValueError(
            "SimCLR external feature label order changed."
        )

    tf.keras.backend.clear_session()
    gc.collect()

    rotation_encoder = build_resnet50_encoder(
        "rotation_resnet50_encoder",
        training_flag=False,
    )
    rotation_encoder.load_weights(
        str(
            ROTATION_SSL_ENCODER_PATH
        )
    )
    rotation_encoder.trainable = False

    (
        rotation_features,
        rotation_feature_labels,
    ) = extract_encoder_features(
        rotation_encoder,
        dataset,
    )

    del rotation_encoder
    gc.collect()

    if not np.array_equal(
        rotation_feature_labels,
        expected_labels,
    ):
        raise ValueError(
            "Rotation external feature label order changed."
        )

    concat_features = np.concatenate(
        [
            simclr_features,
            rotation_features,
        ],
        axis=1,
    ).astype(np.float32)

    # --------------------------------------------------------
    # 3. Standalone Rotation.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    rotation_model = (
        build_rotation_finetune_model()
    )
    rotation_model.load_weights(
        str(ROTATION_FT_PATH)
    )

    (
        rotation_probability,
        rotation_labels,
    ) = predict_direct_model(
        rotation_model,
        dataset,
    )

    del rotation_model
    gc.collect()

    if not np.array_equal(
        rotation_labels,
        expected_labels,
    ):
        raise ValueError(
            "Standalone Rotation external label order changed."
        )

    # --------------------------------------------------------
    # 4. FusionConcat.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    fusionconcat_model = (
        build_fusionconcat_model(
            concat_features.shape[1]
        )
    )
    fusionconcat_model.load_weights(
        str(FUSIONCONCAT_PATH)
    )

    fusionconcat_probability = (
        normalize_probabilities(
            fusionconcat_model.predict(
                concat_features,
                batch_size=256,
                verbose=0,
            )
        )
    )

    del fusionconcat_model
    gc.collect()

    # --------------------------------------------------------
    # 5. FusionGated V2.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    fusiongated_v2_model = (
        build_fusiongated_v2_model(
            simclr_features.shape[1],
            rotation_features.shape[1],
        )
    )
    fusiongated_v2_model.load_weights(
        str(FUSIONGATED_V2_PATH)
    )

    fusiongated_v2_probability = (
        normalize_probabilities(
            fusiongated_v2_model.predict(
                [
                    simclr_features,
                    rotation_features,
                ],
                batch_size=256,
                verbose=0,
            )
        )
    )

    del fusiongated_v2_model
    gc.collect()

    # --------------------------------------------------------
    # 6. FusionGated V3.
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    fusiongated_v3_model = (
        build_fusiongated_v3_model(
            simclr_features.shape[1],
            rotation_features.shape[1],
        )
    )
    fusiongated_v3_model.load_weights(
        str(FUSIONGATED_V3_PATH)
    )

    fusiongated_v3_probability = (
        normalize_probabilities(
            fusiongated_v3_model.predict(
                [
                    simclr_features,
                    rotation_features,
                ],
                batch_size=256,
                verbose=0,
            )
        )
    )

    del fusiongated_v3_model
    gc.collect()

    # --------------------------------------------------------
    # 7. Residual Fusion multi-seed.
    # --------------------------------------------------------
    representative_logits = np.log(
        np.clip(
            representative_probability,
            1e-7,
            1.0,
        )
    ).astype(np.float32)

    residual_probabilities = []

    for seed, weight_path in (
        RESIDUAL_SEED_PATHS.items()
    ):
        print(
            f"Residual Fusion external inference, seed {seed}"
        )

        tf.keras.backend.clear_session()
        gc.collect()

        residual_model = (
            build_residual_fusion_model(
                simclr_features.shape[1],
                rotation_features.shape[1],
            )
        )
        residual_model.load_weights(
            str(weight_path)
        )

        residual_probabilities.append(
            normalize_probabilities(
                residual_model.predict(
                    [
                        simclr_features,
                        rotation_features,
                        representative_logits,
                    ],
                    batch_size=256,
                    verbose=0,
                )
            )
        )

        del residual_model
        gc.collect()

    residual_probability = (
        normalize_probabilities(
            np.mean(
                np.stack(
                    residual_probabilities,
                    axis=0,
                ),
                axis=0,
            )
        )
    )

    source_probabilities = OrderedDict(
        [
            (
                "SimCLR_NoTTA",
                simclr_no_tta_probability,
            ),
            (
                "SimCLR_SelectedTTA",
                simclr_selected_tta_probability,
            ),
            (
                "Rotation",
                rotation_probability,
            ),
            (
                "FusionConcat",
                fusionconcat_probability,
            ),
            (
                "FusionGatedV2",
                fusiongated_v2_probability,
            ),
            (
                "FusionGatedV3",
                fusiongated_v3_probability,
            ),
            (
                "ResidualFusion",
                residual_probability,
            ),
        ]
    )

    if list(
        source_probabilities.keys()
    ) != MODEL_NAMES:
        raise ValueError(
            "External source ordering does not match "
            "the frozen stacking model."
        )

    save_payload = {
        "model_names": np.asarray(
            MODEL_NAMES
        ),
        "labels": expected_labels,
    }

    for model_name, probability in (
        source_probabilities.items()
    ):
        save_payload[
            f"prob__{model_name}"
        ] = probability.astype(
            np.float32
        )

    np.savez_compressed(
        cache_path,
        **save_payload,
    )

    print(
        "Saved external base-probability cache:",
        cache_path,
    )

    return source_probabilities


In [19]:
# ============================================================
# EXTERNAL CELL E07: APPLY BOTH FROZEN STACKING VARIANTS
# ============================================================

def apply_both_frozen_stacking_models(
    raw_source_probabilities,
):
    calibrated_sources = OrderedDict()

    for model_name in MODEL_NAMES:
        calibrated_sources[
            model_name
        ] = temperature_scale(
            raw_source_probabilities[
                model_name
            ],
            TEMPERATURES[
                model_name
            ],
        )

    meta_features = build_meta_features(
        calibrated_sources
    )

    raw_stacking_probability = (
        FINAL_META_MODEL.predict_proba(
            meta_features
        )
    )

    stacking_probability = align_probability(
        raw_stacking_probability,
        FINAL_META_MODEL.named_steps[
            "classifier"
        ].classes_,
    )

    no_bias_probability = (
        normalize_probabilities(
            (
                1.0 - BEST_ALPHA
            )
            * calibrated_sources[
                "SimCLR_NoTTA"
            ]
            + BEST_ALPHA
            * stacking_probability
        )
    )

    logits = np.log(
        np.clip(
            no_bias_probability,
            1e-12,
            1.0,
        )
    )

    logits = (
        logits
        + np.asarray(
            CLASS_BIAS,
            dtype=np.float64,
        )[None, :]
    )

    logits = (
        logits
        - logits.max(
            axis=1,
            keepdims=True,
        )
    )

    class_bias_probability = (
        normalize_probabilities(
            np.exp(logits)
        )
    )

    return {
        NO_BIAS_METHOD: no_bias_probability,
        CLASS_BIAS_METHOD: (
            class_bias_probability
        ),
    }


In [20]:
# ============================================================
# EXTERNAL CELL E08: EXTERNAL METRICS AND OUTPUT HELPERS
# ============================================================

def external_macro_specificity(
    confusion,
):
    total = confusion.sum()
    specificities = []

    for class_id in range(
        confusion.shape[0]
    ):
        true_positive = confusion[
            class_id,
            class_id,
        ]
        false_negative = (
            confusion[
                class_id,
                :,
            ].sum()
            - true_positive
        )
        false_positive = (
            confusion[
                :,
                class_id,
            ].sum()
            - true_positive
        )
        true_negative = (
            total
            - true_positive
            - false_negative
            - false_positive
        )

        denominator = (
            true_negative
            + false_positive
        )

        specificities.append(
            true_negative / denominator
            if denominator > 0
            else 0.0
        )

    return float(
        np.mean(
            specificities
        )
    )


def evaluate_external_probability(
    y_true,
    probability,
    dataset_name,
    method_name,
    class_names=CLASS_NAMES,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    )

    probability = (
        normalize_probabilities(
            probability
        )
    )

    labeled_mask = y_true >= 0

    if np.sum(
        labeled_mask
    ) == 0:
        return None

    y_labeled = y_true[
        labeled_mask
    ]

    probability_labeled = probability[
        labeled_mask
    ]

    prediction = np.argmax(
        probability_labeled,
        axis=1,
    )

    class_ids = np.arange(
        probability_labeled.shape[1]
    )

    confusion = confusion_matrix(
        y_labeled,
        prediction,
        labels=class_ids,
    )

    y_binary = label_binarize(
        y_labeled,
        classes=class_ids,
    )

    try:
        auc = float(
            roc_auc_score(
                y_binary,
                probability_labeled,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        auc = float("nan")

    metrics = {
        "dataset": dataset_name,
        "method": method_name,
        "n_total": int(
            len(y_true)
        ),
        "n_labeled": int(
            np.sum(
                labeled_mask
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_labeled,
                prediction,
            )
        ),
        "bacc": float(
            balanced_accuracy_score(
                y_labeled,
                prediction,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_labeled,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_labeled,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_labeled,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_labeled,
                prediction,
                average="weighted",
                zero_division=0,
            )
        ),
        "macro_specificity": (
            external_macro_specificity(
                confusion
            )
        ),
        "auc_macro_ovr": auc,
    }

    report = classification_report(
        y_labeled,
        prediction,
        labels=class_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )

    return {
        "metrics": metrics,
        "confusion": confusion,
        "report": report,
        "prediction_labeled": prediction,
        "labeled_mask": labeled_mask,
    }


def save_external_method_outputs(
    dataset_name,
    dataframe,
    method_name,
    probability,
    result,
):
    method_dir = (
        EXTERNAL_OUTPUT_DIR
        / dataset_name
        / method_name
    )

    method_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    probability = (
        normalize_probabilities(
            probability
        )
    )

    prediction = np.argmax(
        probability,
        axis=1,
    )

    output_df = dataframe.copy()
    output_df["method"] = method_name
    output_df["pred_label"] = prediction
    output_df["pred_label_name"] = [
        CLASS_NAMES[
            class_id
        ]
        for class_id in prediction
    ]
    output_df["pred_confidence"] = np.max(
        probability,
        axis=1,
    )

    for class_id, class_name in enumerate(
        CLASS_NAMES
    ):
        output_df[
            f"prob_{class_id}_{class_name}"
        ] = probability[
            :,
            class_id,
        ]

    output_df.to_csv(
        method_dir
        / "predictions.csv",
        index=False,
    )

    if result is not None:
        pd.DataFrame(
            result[
                "confusion"
            ],
            index=[
                f"true_{name}"
                for name in CLASS_NAMES
            ],
            columns=[
                f"pred_{name}"
                for name in CLASS_NAMES
            ],
        ).to_csv(
            method_dir
            / "confusion_matrix.csv"
        )

        pd.DataFrame(
            result[
                "report"
            ]
        ).transpose().to_csv(
            method_dir
            / "classification_report.csv"
        )

        with open(
            method_dir
            / "metrics.json",
            "w",
        ) as file:
            json.dump(
                result[
                    "metrics"
                ],
                file,
                indent=2,
            )


In [21]:
# ============================================================
# EXTERNAL CELL E09: RUN BOTH MODELS ON ISIC AND PAD
# ============================================================

EXTERNAL_EIGHT_CLASS_ROWS = []
EXTERNAL_METHOD_PROBABILITIES = {}

for dataset_name, payload in (
    EXTERNAL_DATASETS_FOR_BOTH_STACKS.items()
):
    print("\n" + "=" * 92)
    print(
        "EXTERNAL DATASET:",
        dataset_name,
    )
    print("=" * 92)

    dataframe = payload[
        "dataframe"
    ].reset_index(drop=True)

    dataset = payload[
        "dataset"
    ]

    y_external = dataframe[
        "label"
    ].astype(np.int32).values

    base_probabilities = (
        generate_external_base_probabilities(
            dataset_name,
            dataset,
            y_external,
        )
    )

    both_probabilities = (
        apply_both_frozen_stacking_models(
            base_probabilities
        )
    )

    EXTERNAL_METHOD_PROBABILITIES[
        dataset_name
    ] = {
        "base": base_probabilities,
        "ensembles": both_probabilities,
        "labels": y_external,
    }

    # Report all base methods and both stacking variants.
    all_methods = OrderedDict(
        [
            *list(
                base_probabilities.items()
            ),
            *list(
                both_probabilities.items()
            ),
        ]
    )

    dataset_rows = []

    for method_name, probability in (
        all_methods.items()
    ):
        result = (
            evaluate_external_probability(
                y_external,
                probability,
                dataset_name,
                method_name,
            )
        )

        if result is not None:
            metrics = result[
                "metrics"
            ]
            dataset_rows.append(
                metrics
            )
            EXTERNAL_EIGHT_CLASS_ROWS.append(
                metrics
            )

        save_external_method_outputs(
            dataset_name,
            dataframe,
            method_name,
            probability,
            result,
        )

    dataset_results_df = pd.DataFrame(
        dataset_rows
    ).sort_values(
        [
            "bacc",
            "macro_f1",
            "accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    ).reset_index(drop=True)

    dataset_results_df[
        "rank_by_bacc"
    ] = np.arange(
        1,
        len(dataset_results_df) + 1,
    )

    dataset_results_df.to_csv(
        EXTERNAL_OUTPUT_DIR
        / dataset_name
        / "all_methods_eight_class_comparison.csv",
        index=False,
    )

    print(
        "\nEight-class/existing-label comparison:"
    )
    display(
        dataset_results_df
    )

EXTERNAL_EIGHT_CLASS_DF = pd.DataFrame(
    EXTERNAL_EIGHT_CLASS_ROWS
)

EXTERNAL_EIGHT_CLASS_DF.to_csv(
    EXTERNAL_OUTPUT_DIR
    / "all_external_eight_class_results.csv",
    index=False,
)



EXTERNAL DATASET: ISIC_2019_Official_Test
Generating external base probabilities for ISIC_2019_Official_Test
SimCLR external inference, seed 21
SimCLR external inference, seed 35
SimCLR external inference, seed 42
SimCLR external inference, seed 52
SimCLR external inference, seed 62


I0000 00:00:1784051565.558811      65 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Residual Fusion external inference, seed 21
Residual Fusion external inference, seed 42
Residual Fusion external inference, seed 62
Saved external base-probability cache: /kaggle/working/both_stacking_models_external_evaluation/base_probability_cache/ISIC_2019_Official_Test_all_base_probabilities.npz

Eight-class/existing-label comparison:


                   dataset                         method  n_total  n_labeled  \
0  ISIC_2019_Official_Test            NoTTA_StackingBlend     8238       6191   
1  ISIC_2019_Official_Test  NoTTA_StackingBlend_ClassBias     8238       6191   
2  ISIC_2019_Official_Test                       Rotation     8238       6191   
3  ISIC_2019_Official_Test                   SimCLR_NoTTA     8238       6191   
4  ISIC_2019_Official_Test             SimCLR_SelectedTTA     8238       6191   
5  ISIC_2019_Official_Test                  FusionGatedV2     8238       6191   
6  ISIC_2019_Official_Test                   FusionConcat     8238       6191   
7  ISIC_2019_Official_Test                  FusionGatedV3     8238       6191   
8  ISIC_2019_Official_Test                 ResidualFusion     8238       6191   

   accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f1  \
0  0.688096  0.570154         0.563016      0.570154  0.563075     0.690113   
1  0.687934  0.568522         0


EXTERNAL DATASET: PAD_UFES_20_External
Generating external base probabilities for PAD_UFES_20_External
SimCLR external inference, seed 21
SimCLR external inference, seed 35
SimCLR external inference, seed 42
SimCLR external inference, seed 52
SimCLR external inference, seed 62
Residual Fusion external inference, seed 21
Residual Fusion external inference, seed 42
Residual Fusion external inference, seed 62
Saved external base-probability cache: /kaggle/working/both_stacking_models_external_evaluation/base_probability_cache/PAD_UFES_20_External_all_base_probabilities.npz


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/


Eight-class/existing-label comparison:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


                dataset                         method  n_total  n_labeled  \
0  PAD_UFES_20_External                  FusionGatedV2     2298       2298   
1  PAD_UFES_20_External  NoTTA_StackingBlend_ClassBias     2298       2298   
2  PAD_UFES_20_External            NoTTA_StackingBlend     2298       2298   
3  PAD_UFES_20_External             SimCLR_SelectedTTA     2298       2298   
4  PAD_UFES_20_External                   SimCLR_NoTTA     2298       2298   
5  PAD_UFES_20_External                  FusionGatedV3     2298       2298   
6  PAD_UFES_20_External                 ResidualFusion     2298       2298   
7  PAD_UFES_20_External                   FusionConcat     2298       2298   
8  PAD_UFES_20_External                       Rotation     2298       2298   

   accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f1  \
0  0.294604  0.331170         0.243318      0.248377  0.188149     0.302711   
1  0.272411  0.330030         0.260222      0.247522  0.18978

In [22]:
# ============================================================
# EXTERNAL CELL E10: PAD SIX-CLASS RESTRICTED COMPARISON
# ============================================================

PAD_COMMON_CLASS_IDS = [
    0,  # MEL
    1,  # NV
    2,  # BCC
    3,  # AK
    4,  # BKL
    7,  # SCC
]

PAD_SIX_CLASS_NAMES = [
    CLASS_NAMES[
        class_id
    ]
    for class_id
    in PAD_COMMON_CLASS_IDS
]

ORIGINAL_TO_PAD6 = {
    original_id: new_id
    for new_id, original_id
    in enumerate(
        PAD_COMMON_CLASS_IDS
    )
}


def restrict_probability_to_pad6(
    y_true,
    probability,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int32,
    )

    probability = normalize_probabilities(
        probability
    )

    valid_mask = np.isin(
        y_true,
        PAD_COMMON_CLASS_IDS,
    )

    y_original = y_true[
        valid_mask
    ]

    y_pad6 = np.asarray(
        [
            ORIGINAL_TO_PAD6[
                int(label)
            ]
            for label in y_original
        ],
        dtype=np.int32,
    )

    probability_pad6 = (
        normalize_probabilities(
            probability[
                valid_mask
            ][
                :,
                PAD_COMMON_CLASS_IDS,
            ]
        )
    )

    return (
        y_pad6,
        probability_pad6,
        valid_mask,
    )


pad_bundle = EXTERNAL_METHOD_PROBABILITIES[
    "PAD_UFES_20_External"
]

pad_y_original = pad_bundle[
    "labels"
]

pad_all_methods = OrderedDict(
    [
        *list(
            pad_bundle[
                "base"
            ].items()
        ),
        *list(
            pad_bundle[
                "ensembles"
            ].items()
        ),
    ]
)

pad_six_rows = []

for method_name, probability in (
    pad_all_methods.items()
):
    (
        y_pad6,
        probability_pad6,
        valid_mask,
    ) = restrict_probability_to_pad6(
        pad_y_original,
        probability,
    )

    prediction_pad6 = np.argmax(
        probability_pad6,
        axis=1,
    )

    confusion_pad6 = confusion_matrix(
        y_pad6,
        prediction_pad6,
        labels=np.arange(6),
    )

    pad_six_rows.append(
        {
            "dataset": (
                "PAD_UFES_20_"
                "SixClassRestricted"
            ),
            "method": method_name,
            "n_samples": int(
                len(y_pad6)
            ),
            "accuracy": float(
                accuracy_score(
                    y_pad6,
                    prediction_pad6,
                )
            ),
            "bacc": float(
                balanced_accuracy_score(
                    y_pad6,
                    prediction_pad6,
                )
            ),
            "macro_precision": float(
                precision_score(
                    y_pad6,
                    prediction_pad6,
                    average="macro",
                    zero_division=0,
                )
            ),
            "macro_recall": float(
                recall_score(
                    y_pad6,
                    prediction_pad6,
                    average="macro",
                    zero_division=0,
                )
            ),
            "macro_f1": float(
                f1_score(
                    y_pad6,
                    prediction_pad6,
                    average="macro",
                    zero_division=0,
                )
            ),
            "weighted_f1": float(
                f1_score(
                    y_pad6,
                    prediction_pad6,
                    average="weighted",
                    zero_division=0,
                )
            ),
            "macro_specificity": (
                external_macro_specificity(
                    confusion_pad6
                )
            ),
        }
    )

PAD_SIX_CLASS_RESULTS_DF = (
    pd.DataFrame(
        pad_six_rows
    ).sort_values(
        [
            "bacc",
            "macro_f1",
            "accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    ).reset_index(drop=True)
)

PAD_SIX_CLASS_RESULTS_DF[
    "rank_by_bacc"
] = np.arange(
    1,
    len(
        PAD_SIX_CLASS_RESULTS_DF
    ) + 1,
)

PAD_SIX_CLASS_RESULTS_DF.to_csv(
    EXTERNAL_OUTPUT_DIR
    / "PAD_UFES_20_External"
    / "pad_six_class_all_methods_comparison.csv",
    index=False,
)

print("=" * 92)
print("PAD SIX-CLASS RESTRICTED COMPARISON")
print("=" * 92)
display(
    PAD_SIX_CLASS_RESULTS_DF
)


PAD SIX-CLASS RESTRICTED COMPARISON


                          dataset                         method  n_samples  \
0  PAD_UFES_20_SixClassRestricted  NoTTA_StackingBlend_ClassBias       2298   
1  PAD_UFES_20_SixClassRestricted                  FusionGatedV2       2298   
2  PAD_UFES_20_SixClassRestricted            NoTTA_StackingBlend       2298   
3  PAD_UFES_20_SixClassRestricted             SimCLR_SelectedTTA       2298   
4  PAD_UFES_20_SixClassRestricted                   FusionConcat       2298   
5  PAD_UFES_20_SixClassRestricted                   SimCLR_NoTTA       2298   
6  PAD_UFES_20_SixClassRestricted                  FusionGatedV3       2298   
7  PAD_UFES_20_SixClassRestricted                 ResidualFusion       2298   
8  PAD_UFES_20_SixClassRestricted                       Rotation       2298   

   accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f1  \
0  0.287641  0.344139         0.345304      0.344139  0.261053     0.294207   
1  0.302437  0.340017         0.325723      0.34001

In [23]:
# ============================================================
# EXTERNAL CELL E11: DIRECT BOTH-STACKING COMPARISON AND EXPORT
# ============================================================

both_method_rows = []

if len(
    EXTERNAL_EIGHT_CLASS_DF
) > 0:
    both_method_rows.extend(
        EXTERNAL_EIGHT_CLASS_DF.loc[
            EXTERNAL_EIGHT_CLASS_DF[
                "method"
            ].isin(
                [
                    NO_BIAS_METHOD,
                    CLASS_BIAS_METHOD,
                ]
            )
        ].to_dict(
            orient="records"
        )
    )

both_method_rows.extend(
    PAD_SIX_CLASS_RESULTS_DF.loc[
        PAD_SIX_CLASS_RESULTS_DF[
            "method"
        ].isin(
            [
                NO_BIAS_METHOD,
                CLASS_BIAS_METHOD,
            ]
        )
    ].to_dict(
        orient="records"
    )
)

BOTH_STACKING_EXTERNAL_DF = pd.DataFrame(
    both_method_rows
)

BOTH_STACKING_EXTERNAL_DF.to_csv(
    EXTERNAL_OUTPUT_DIR
    / "BOTH_STACKING_MODELS_EXTERNAL_RESULTS.csv",
    index=False,
)

print("=" * 92)
print("BOTH FROZEN STACKING MODELS — EXTERNAL RESULTS")
print("=" * 92)
display(
    BOTH_STACKING_EXTERNAL_DF
)

print(
    "\nInterpretation rule:"
)
print(
    "- NoTTA_StackingBlend_ClassBias remains the proposed final "
    "model because internal validation selected it."
)
print(
    "- NoTTA_StackingBlend is reported as the secondary "
    "no-bias ensemble/ablation."
)
print(
    "- Do not change the proposed final model according to "
    "ISIC or PAD test BACC."
)

EXTERNAL_FINAL_SUMMARY = {
    "proposed_final_model": (
        CLASS_BIAS_METHOD
    ),
    "secondary_comparison_model": (
        NO_BIAS_METHOD
    ),
    "selection_source": (
        "internal validation only"
    ),
    "external_test_used_for_selection": False,
    "frozen_alpha": float(
        BEST_ALPHA
    ),
    "frozen_class_bias": [
        float(value)
        for value in np.asarray(
            CLASS_BIAS
        )
    ],
    "frozen_temperatures": {
        model_name: float(
            TEMPERATURES[
                model_name
            ]
        )
        for model_name
        in MODEL_NAMES
    },
}

with open(
    EXTERNAL_OUTPUT_DIR
    / "external_frozen_ensemble_summary.json",
    "w",
) as file:
    json.dump(
        EXTERNAL_FINAL_SUMMARY,
        file,
        indent=2,
    )

PACKAGE_BASE = Path(
    "/kaggle/working/"
    "both_stacking_models_external_evaluation"
)

ARCHIVE_PATH = shutil.make_archive(
    base_name=str(
        PACKAGE_BASE
    ),
    format="zip",
    root_dir=str(
        EXTERNAL_OUTPUT_DIR
    ),
)

print(
    "\nFinal external output directory:"
)
print(
    EXTERNAL_OUTPUT_DIR
)

print(
    "\nDownloadable external-results ZIP:"
)
print(
    ARCHIVE_PATH
)

print(
    "\nNow click Kaggle Save Version."
)


BOTH FROZEN STACKING MODELS — EXTERNAL RESULTS


                          dataset                         method  n_total  \
0         ISIC_2019_Official_Test            NoTTA_StackingBlend   8238.0   
1         ISIC_2019_Official_Test  NoTTA_StackingBlend_ClassBias   8238.0   
2            PAD_UFES_20_External            NoTTA_StackingBlend   2298.0   
3            PAD_UFES_20_External  NoTTA_StackingBlend_ClassBias   2298.0   
4  PAD_UFES_20_SixClassRestricted  NoTTA_StackingBlend_ClassBias      NaN   
5  PAD_UFES_20_SixClassRestricted            NoTTA_StackingBlend      NaN   

   n_labeled  accuracy      bacc  macro_precision  macro_recall  macro_f1  \
0     6191.0  0.688096  0.570154         0.563016      0.570154  0.563075   
1     6191.0  0.687934  0.568522         0.565910      0.568522  0.562874   
2     2298.0  0.269365  0.326400         0.255758      0.244800  0.187349   
3     2298.0  0.272411  0.330030         0.260222      0.247522  0.189784   
4        NaN  0.287641  0.344139         0.345304      0.344139  0.261053  


Interpretation rule:
- NoTTA_StackingBlend_ClassBias remains the proposed final model because internal validation selected it.
- NoTTA_StackingBlend is reported as the secondary no-bias ensemble/ablation.
- Do not change the proposed final model according to ISIC or PAD test BACC.

Final external output directory:
/kaggle/working/both_stacking_models_external_evaluation

Downloadable external-results ZIP:
/kaggle/working/both_stacking_models_external_evaluation.zip

Now click Kaggle Save Version.


# PAD-UFES-20 Fine-Tuned / Domain-Adapted Ensemble Extension

This section creates PAD-specific six-class versions of both final stacking
models without retraining the SSL encoders or feature-fusion networks.

## Domain-adaptation protocol

- Reuse the seven PAD base-model probability arrays already generated above.
- Reuse the original PAD 70/15/15 split: 1,608 training, 345 validation,
  and 345 test samples.
- Restrict predictions to the six PAD/ISIC common classes:
  `MEL, NV, BCC, AK, BKL, SCC`.
- Calibrate every base model using PAD training labels only.
- Train a class-balanced six-class logistic stacking adapter on PAD training.
- Select `C` and the no-TTA residual-blend strength using PAD validation only.
- Select optional six-class logit biases using PAD validation only.
- Evaluate the two frozen domain-adapted variants once on PAD test.

The two reported adapted models are:

1. **PAD-DA NoTTA Stacking Blend**
2. **PAD-DA NoTTA Stacking Blend + Class Bias**

This is a PAD fine-tuned/domain-adapted experiment, not a zero-shot result.


In [24]:
# ============================================================
# PAD-DA CELL D01: SAFETY CHECKS AND OUTPUT DIRECTORY
# ============================================================

import json
import shutil
from collections import OrderedDict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

PAD_DA_SEED = 42
PAD_DA_RNG = np.random.default_rng(
    PAD_DA_SEED
)

PAD_DA_OUTPUT_DIR = (
    Path("/kaggle/working")
    / "pad_domain_adapted_both_stacking_models"
)
PAD_DA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PAD_DA_CLASS_IDS_8 = [
    0,  # MEL
    1,  # NV
    2,  # BCC
    3,  # AK
    4,  # BKL
    7,  # SCC
]

PAD_DA_CLASS_NAMES = [
    "MEL",
    "NV",
    "BCC",
    "AK",
    "BKL",
    "SCC",
]

PAD_DA_NUM_CLASSES = len(
    PAD_DA_CLASS_NAMES
)

PAD_DA_MODEL_NAMES = list(
    MODEL_NAMES
)

required_pad_da_variables = [
    "PIPELINE_ROOT",
    "PAD_EXTERNAL_DF",
    "EXTERNAL_METHOD_PROBABILITIES",
    "MODEL_NAMES",
    "normalize_probabilities",
]

missing_pad_da_variables = [
    name
    for name in required_pad_da_variables
    if name not in globals()
]

if missing_pad_da_variables:
    raise ValueError(
        "Run the internal + external notebook cells first. "
        "Missing variables: "
        f"{missing_pad_da_variables}"
    )

if (
    "PAD_UFES_20_External"
    not in EXTERNAL_METHOD_PROBABILITIES
):
    raise ValueError(
        "PAD external probabilities were not generated."
    )

PAD_DA_BASE_PROB_8 = OrderedDict(
    EXTERNAL_METHOD_PROBABILITIES[
        "PAD_UFES_20_External"
    ]["base"]
)

PAD_DA_ZERO_SHOT_ENSEMBLES_8 = OrderedDict(
    EXTERNAL_METHOD_PROBABILITIES[
        "PAD_UFES_20_External"
    ]["ensembles"]
)

if list(
    PAD_DA_BASE_PROB_8.keys()
) != PAD_DA_MODEL_NAMES:
    raise ValueError(
        "PAD base-model probability order differs from "
        "the internal stacking order."
    )

print("=" * 92)
print("PAD SIX-CLASS META-LEVEL DOMAIN ADAPTATION")
print("=" * 92)
print("Base models:", PAD_DA_MODEL_NAMES)
print("Output directory:", PAD_DA_OUTPUT_DIR)


PAD SIX-CLASS META-LEVEL DOMAIN ADAPTATION
Base models: ['SimCLR_NoTTA', 'SimCLR_SelectedTTA', 'Rotation', 'FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']
Output directory: /kaggle/working/pad_domain_adapted_both_stacking_models


In [25]:
# ============================================================
# PAD-DA CELL D02: REUSE THE ORIGINAL 70/15/15 PAD SPLIT
# ============================================================

def pad_da_image_key(value):
    return Path(
        str(value)
    ).stem.lower().strip()


pad_da_df = PAD_EXTERNAL_DF.copy().reset_index(
    drop=True
)

if "image_id" in pad_da_df.columns:
    pad_da_df["pad_da_key"] = pad_da_df[
        "image_id"
    ].map(
        pad_da_image_key
    )
elif "image_path_current" in pad_da_df.columns:
    pad_da_df["pad_da_key"] = pad_da_df[
        "image_path_current"
    ].map(
        pad_da_image_key
    )
else:
    raise ValueError(
        "PAD dataframe has no image_id or image_path_current."
    )

pad_da_df["label"] = pad_da_df[
    "label"
].astype(np.int32)

pad_da_df = pad_da_df.loc[
    pad_da_df["label"].isin(
        PAD_DA_CLASS_IDS_8
    )
].copy().reset_index(drop=True)

PAD_DA_ORIGINAL_TO_6 = {
    original_id: local_id
    for local_id, original_id
    in enumerate(
        PAD_DA_CLASS_IDS_8
    )
}

pad_da_df["label_6"] = pad_da_df[
    "label"
].map(
    PAD_DA_ORIGINAL_TO_6
).astype(np.int32)

# The old notebook saved the exact split used for the
# previously reported PAD fine-tuned FusionConcat result.
PAD_DA_PREVIOUS_DIR = (
    Path(PIPELINE_ROOT)
    / "pad_ufes_20_six_class_finetuning"
)

PAD_DA_PREVIOUS_SPLIT_CSV = (
    PAD_DA_PREVIOUS_DIR
    / "pad_six_class_split.csv"
)

PAD_DA_USED_PREVIOUS_SPLIT = False

if PAD_DA_PREVIOUS_SPLIT_CSV.exists():
    previous_split_df = pd.read_csv(
        PAD_DA_PREVIOUS_SPLIT_CSV
    )

    if "image_id" in previous_split_df.columns:
        previous_split_df[
            "pad_da_key"
        ] = previous_split_df[
            "image_id"
        ].map(
            pad_da_image_key
        )
    elif "image_path" in previous_split_df.columns:
        previous_split_df[
            "pad_da_key"
        ] = previous_split_df[
            "image_path"
        ].map(
            pad_da_image_key
        )
    elif "image_path_current" in previous_split_df.columns:
        previous_split_df[
            "pad_da_key"
        ] = previous_split_df[
            "image_path_current"
        ].map(
            pad_da_image_key
        )
    else:
        raise ValueError(
            "The saved PAD split has no usable image identifier."
        )

    if "split" not in previous_split_df.columns:
        raise ValueError(
            "The saved PAD split has no split column."
        )

    split_map_df = previous_split_df[
        [
            "pad_da_key",
            "split",
        ]
    ].drop_duplicates(
        subset=["pad_da_key"]
    )

    pad_da_df = pad_da_df.merge(
        split_map_df,
        on="pad_da_key",
        how="left",
        validate="one_to_one",
    )

    missing_split_count = int(
        pad_da_df["split"].isna().sum()
    )

    if missing_split_count == 0:
        PAD_DA_USED_PREVIOUS_SPLIT = True
    else:
        print(
            "Saved split could not map all images; "
            "recreating the deterministic split. "
            "Unmapped:",
            missing_split_count,
        )
        pad_da_df = pad_da_df.drop(
            columns=["split"]
        )

if not PAD_DA_USED_PREVIOUS_SPLIT:
    pad_da_train_df, pad_da_temp_df = (
        train_test_split(
            pad_da_df,
            test_size=0.30,
            stratify=pad_da_df[
                "label_6"
            ],
            random_state=PAD_DA_SEED,
        )
    )

    pad_da_val_df, pad_da_test_df = (
        train_test_split(
            pad_da_temp_df,
            test_size=0.50,
            stratify=pad_da_temp_df[
                "label_6"
            ],
            random_state=PAD_DA_SEED,
        )
    )

    pad_da_df["split"] = ""

    pad_da_df.loc[
        pad_da_train_df.index,
        "split",
    ] = "train"

    pad_da_df.loc[
        pad_da_val_df.index,
        "split",
    ] = "val"

    pad_da_df.loc[
        pad_da_test_df.index,
        "split",
    ] = "test"

valid_split_names = {
    "train",
    "val",
    "test",
}

if set(
    pad_da_df["split"].unique()
) != valid_split_names:
    raise ValueError(
        "PAD split must contain exactly train, val, and test."
    )

PAD_DA_TRAIN_INDEX = np.flatnonzero(
    pad_da_df["split"].values == "train"
)
PAD_DA_VAL_INDEX = np.flatnonzero(
    pad_da_df["split"].values == "val"
)
PAD_DA_TEST_INDEX = np.flatnonzero(
    pad_da_df["split"].values == "test"
)

PAD_DA_Y_ALL = pad_da_df[
    "label_6"
].to_numpy(
    dtype=np.int32
)

PAD_DA_Y_TRAIN = PAD_DA_Y_ALL[
    PAD_DA_TRAIN_INDEX
]
PAD_DA_Y_VAL = PAD_DA_Y_ALL[
    PAD_DA_VAL_INDEX
]
PAD_DA_Y_TEST = PAD_DA_Y_ALL[
    PAD_DA_TEST_INDEX
]

split_summary = (
    pad_da_df.groupby(
        [
            "split",
            "label_6",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print(
    "Reused previous exact PAD split:",
    PAD_DA_USED_PREVIOUS_SPLIT,
)
print(
    "Train / validation / test:",
    len(PAD_DA_TRAIN_INDEX),
    len(PAD_DA_VAL_INDEX),
    len(PAD_DA_TEST_INDEX),
)
display(split_summary)

if (
    len(PAD_DA_TRAIN_INDEX) != 1608
    or len(PAD_DA_VAL_INDEX) != 345
    or len(PAD_DA_TEST_INDEX) != 345
):
    print(
        "Warning: counts differ from the original "
        "1608/345/345 split."
    )

pad_da_df.to_csv(
    PAD_DA_OUTPUT_DIR
    / "pad_domain_adaptation_split.csv",
    index=False,
)


Reused previous exact PAD split: True
Train / validation / test: 1608 345 345


label_6   0    1    2    3    4    5
split                               
test      8   37  127  109   35   29
train    36  171  591  511  165  134
val       8   36  127  110   35   29

In [26]:
# ============================================================
# PAD-DA CELL D03: SIX-CLASS PROBABILITIES AND TRAIN CALIBRATION
# ============================================================

def pad_da_restrict_to_six(
    probability_8,
):
    probability_8 = (
        normalize_probabilities(
            probability_8
        )
    )

    return normalize_probabilities(
        probability_8[
            :,
            PAD_DA_CLASS_IDS_8,
        ]
    )


PAD_DA_RAW_PROB_6 = OrderedDict(
    [
        (
            model_name,
            pad_da_restrict_to_six(
                PAD_DA_BASE_PROB_8[
                    model_name
                ]
            ),
        )
        for model_name
        in PAD_DA_MODEL_NAMES
    ]
)

PAD_DA_TEMPERATURE_GRID = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00,
    1.10,
    1.20,
    1.30,
    1.50,
    1.75,
    2.00,
    2.25,
    2.50,
]


def pad_da_temperature_scale(
    probability,
    temperature,
):
    probability = normalize_probabilities(
        probability
    )

    logits = np.log(
        np.clip(
            probability,
            1e-12,
            1.0,
        )
    )
    logits /= float(
        temperature
    )
    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    return normalize_probabilities(
        np.exp(logits)
    )


PAD_DA_TEMPERATURES = {}
PAD_DA_CALIBRATED_PROB_6 = OrderedDict()
pad_da_temperature_rows = []

for model_name, probability in (
    PAD_DA_RAW_PROB_6.items()
):
    candidate_rows = []

    for temperature in (
        PAD_DA_TEMPERATURE_GRID
    ):
        calibrated_train = (
            pad_da_temperature_scale(
                probability[
                    PAD_DA_TRAIN_INDEX
                ],
                temperature,
            )
        )

        candidate_rows.append(
            {
                "temperature": float(
                    temperature
                ),
                "train_nll": float(
                    log_loss(
                        PAD_DA_Y_TRAIN,
                        calibrated_train,
                        labels=np.arange(
                            PAD_DA_NUM_CLASSES
                        ),
                    )
                ),
            }
        )

    candidate_df = pd.DataFrame(
        candidate_rows
    ).sort_values(
        [
            "train_nll",
            "temperature",
        ],
        ascending=[
            True,
            True,
        ],
    )

    selected_row = candidate_df.iloc[
        0
    ]

    selected_temperature = float(
        selected_row["temperature"]
    )

    PAD_DA_TEMPERATURES[
        model_name
    ] = selected_temperature

    PAD_DA_CALIBRATED_PROB_6[
        model_name
    ] = pad_da_temperature_scale(
        probability,
        selected_temperature,
    )

    pad_da_temperature_rows.append(
        {
            "model": model_name,
            "selected_temperature": (
                selected_temperature
            ),
            "train_nll": float(
                selected_row[
                    "train_nll"
                ]
            ),
        }
    )

PAD_DA_TEMPERATURE_DF = pd.DataFrame(
    pad_da_temperature_rows
)

display(
    PAD_DA_TEMPERATURE_DF
)

PAD_DA_TEMPERATURE_DF.to_csv(
    PAD_DA_OUTPUT_DIR
    / "pad_train_temperature_scaling.csv",
    index=False,
)


                model  selected_temperature  train_nll
0        SimCLR_NoTTA                   2.5   2.383741
1  SimCLR_SelectedTTA                   2.5   2.346882
2            Rotation                   2.5   3.080696
3        FusionConcat                   2.5   1.944466
4       FusionGatedV2                   2.5   1.831112
5       FusionGatedV3                   2.5   1.944520
6      ResidualFusion                   2.5   3.246794

In [27]:
# ============================================================
# PAD-DA CELL D04: TRAIN/VALIDATION META-ADAPTER SEARCH
# ============================================================

def pad_da_entropy(
    probability,
):
    probability = np.clip(
        probability,
        1e-12,
        1.0,
    )

    return -np.sum(
        probability
        * np.log(
            probability
        ),
        axis=1,
        keepdims=True,
    )


def pad_da_build_meta_features(
    source_probabilities,
):
    blocks = []

    for model_name in (
        PAD_DA_MODEL_NAMES
    ):
        probability = (
            source_probabilities[
                model_name
            ]
        )

        blocks.extend(
            [
                probability,
                np.log(
                    np.clip(
                        probability,
                        1e-12,
                        1.0,
                    )
                ),
                np.max(
                    probability,
                    axis=1,
                    keepdims=True,
                ),
                pad_da_entropy(
                    probability
                ),
            ]
        )

    return np.concatenate(
        blocks,
        axis=1,
    ).astype(np.float32)


def pad_da_make_meta_model(
    c_value,
):
    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=float(
                        c_value
                    ),
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=PAD_DA_SEED,
                ),
            ),
        ]
    )


def pad_da_align_probability(
    raw_probability,
    classes_seen,
):
    aligned = np.zeros(
        (
            len(raw_probability),
            PAD_DA_NUM_CLASSES,
        ),
        dtype=np.float32,
    )

    for source_column, class_id in enumerate(
        classes_seen
    ):
        aligned[
            :,
            int(class_id),
        ] = raw_probability[
            :,
            source_column,
        ]

    return normalize_probabilities(
        aligned
    )


PAD_DA_META_X_ALL = (
    pad_da_build_meta_features(
        PAD_DA_CALIBRATED_PROB_6
    )
)

PAD_DA_C_VALUES = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.10,
    0.30,
    1.00,
    3.00,
    10.00,
]

PAD_DA_ALPHA_VALUES = np.round(
    np.arange(
        0.00,
        1.01,
        0.05,
    ),
    2,
)

PAD_DA_ANCHOR_ALL = (
    PAD_DA_CALIBRATED_PROB_6[
        "SimCLR_NoTTA"
    ]
)

pad_da_search_rows = []
pad_da_models_by_c = {}

for c_value in PAD_DA_C_VALUES:
    model = pad_da_make_meta_model(
        c_value
    )

    model.fit(
        PAD_DA_META_X_ALL[
            PAD_DA_TRAIN_INDEX
        ],
        PAD_DA_Y_TRAIN,
    )

    raw_val_probability = (
        model.predict_proba(
            PAD_DA_META_X_ALL[
                PAD_DA_VAL_INDEX
            ]
        )
    )

    val_probability = (
        pad_da_align_probability(
            raw_val_probability,
            model.named_steps[
                "classifier"
            ].classes_,
        )
    )

    pad_da_models_by_c[
        float(c_value)
    ] = model

    for alpha in (
        PAD_DA_ALPHA_VALUES
    ):
        blended_val_probability = (
            normalize_probabilities(
                (
                    1.0 - alpha
                )
                * PAD_DA_ANCHOR_ALL[
                    PAD_DA_VAL_INDEX
                ]
                + alpha
                * val_probability
            )
        )

        val_prediction = np.argmax(
            blended_val_probability,
            axis=1,
        )

        pad_da_search_rows.append(
            {
                "C": float(
                    c_value
                ),
                "alpha_stack": float(
                    alpha
                ),
                "val_bacc": float(
                    balanced_accuracy_score(
                        PAD_DA_Y_VAL,
                        val_prediction,
                    )
                ),
                "val_macro_f1": float(
                    f1_score(
                        PAD_DA_Y_VAL,
                        val_prediction,
                        average="macro",
                        zero_division=0,
                    )
                ),
                "val_accuracy": float(
                    accuracy_score(
                        PAD_DA_Y_VAL,
                        val_prediction,
                    )
                ),
                "val_nll": float(
                    log_loss(
                        PAD_DA_Y_VAL,
                        blended_val_probability,
                        labels=np.arange(
                            PAD_DA_NUM_CLASSES
                        ),
                    )
                ),
            }
        )

PAD_DA_STACK_SEARCH_DF = pd.DataFrame(
    pad_da_search_rows
).sort_values(
    [
        "val_bacc",
        "val_macro_f1",
        "val_accuracy",
        "val_nll",
        "alpha_stack",
        "C",
    ],
    ascending=[
        False,
        False,
        False,
        True,
        True,
        True,
    ],
).reset_index(drop=True)

PAD_DA_BEST_STACK_ROW = (
    PAD_DA_STACK_SEARCH_DF.iloc[0]
)

PAD_DA_BEST_C = float(
    PAD_DA_BEST_STACK_ROW["C"]
)

PAD_DA_BEST_ALPHA = float(
    PAD_DA_BEST_STACK_ROW[
        "alpha_stack"
    ]
)

PAD_DA_META_MODEL = (
    pad_da_models_by_c[
        PAD_DA_BEST_C
    ]
)

raw_pad_da_val_stack = (
    PAD_DA_META_MODEL.predict_proba(
        PAD_DA_META_X_ALL[
            PAD_DA_VAL_INDEX
        ]
    )
)

PAD_DA_VAL_STACK_PROB = (
    pad_da_align_probability(
        raw_pad_da_val_stack,
        PAD_DA_META_MODEL.named_steps[
            "classifier"
        ].classes_,
    )
)

PAD_DA_NO_BIAS_VAL_PROB = (
    normalize_probabilities(
        (
            1.0 - PAD_DA_BEST_ALPHA
        )
        * PAD_DA_ANCHOR_ALL[
            PAD_DA_VAL_INDEX
        ]
        + PAD_DA_BEST_ALPHA
        * PAD_DA_VAL_STACK_PROB
    )
)

PAD_DA_STACK_SEARCH_DF.to_csv(
    PAD_DA_OUTPUT_DIR
    / "pad_meta_adapter_validation_search.csv",
    index=False,
)

joblib.dump(
    PAD_DA_META_MODEL,
    PAD_DA_OUTPUT_DIR
    / "pad_six_class_stacking_adapter.joblib",
)

print("=" * 92)
print("PAD META-ADAPTER VALIDATION SELECTION")
print("=" * 92)
print("Selected C:", PAD_DA_BEST_C)
print(
    "Selected stacking alpha:",
    PAD_DA_BEST_ALPHA,
)
display(
    PAD_DA_BEST_STACK_ROW.to_frame().T
)


PAD META-ADAPTER VALIDATION SELECTION
Selected C: 0.3
Selected stacking alpha: 0.65


     C  alpha_stack  val_bacc  val_macro_f1  val_accuracy   val_nll
0  0.3         0.65  0.519407      0.448482      0.515942  1.333829

In [28]:
# ============================================================
# PAD-DA CELL D05: VALIDATION-ONLY SIX-CLASS BIAS SEARCH
# ============================================================

PAD_DA_BIAS_VALUES = [
    -0.30,
    -0.20,
    -0.15,
    -0.10,
    -0.05,
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.30,
]


def pad_da_apply_bias(
    probability,
    bias,
):
    logits = np.log(
        np.clip(
            normalize_probabilities(
                probability
            ),
            1e-12,
            1.0,
        )
    )

    logits += np.asarray(
        bias,
        dtype=np.float64,
    )[None, :]

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    return normalize_probabilities(
        np.exp(logits)
    )


def pad_da_validation_tuple(
    probability,
    bias=None,
):
    prediction = np.argmax(
        probability,
        axis=1,
    )

    regularization = (
        0.0
        if bias is None
        else 1e-4
        * float(
            np.sum(
                np.abs(
                    bias
                )
            )
        )
    )

    return (
        float(
            balanced_accuracy_score(
                PAD_DA_Y_VAL,
                prediction,
            )
        )
        - regularization,
        float(
            f1_score(
                PAD_DA_Y_VAL,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        float(
            accuracy_score(
                PAD_DA_Y_VAL,
                prediction,
            )
        ),
        -float(
            log_loss(
                PAD_DA_Y_VAL,
                probability,
                labels=np.arange(
                    PAD_DA_NUM_CLASSES
                ),
            )
        ),
    )


PAD_DA_CLASS_BIAS = np.zeros(
    PAD_DA_NUM_CLASSES,
    dtype=np.float32,
)

for _ in range(3):
    for class_id in range(
        PAD_DA_NUM_CLASSES
    ):
        best_local = None

        for candidate_bias in (
            PAD_DA_BIAS_VALUES
        ):
            trial_bias = (
                PAD_DA_CLASS_BIAS.copy()
            )
            trial_bias[
                class_id
            ] = candidate_bias

            trial_probability = (
                pad_da_apply_bias(
                    PAD_DA_NO_BIAS_VAL_PROB,
                    trial_bias,
                )
            )

            score = (
                *pad_da_validation_tuple(
                    trial_probability,
                    trial_bias,
                ),
                -abs(
                    candidate_bias
                ),
            )

            if (
                best_local is None
                or score > best_local[0]
            ):
                best_local = (
                    score,
                    candidate_bias,
                )

        PAD_DA_CLASS_BIAS[
            class_id
        ] = best_local[1]

PAD_DA_CLASS_BIAS_VAL_PROB = (
    pad_da_apply_bias(
        PAD_DA_NO_BIAS_VAL_PROB,
        PAD_DA_CLASS_BIAS,
    )
)

PAD_DA_NO_BIAS_VAL_PRED = np.argmax(
    PAD_DA_NO_BIAS_VAL_PROB,
    axis=1,
)

PAD_DA_CLASS_BIAS_VAL_PRED = np.argmax(
    PAD_DA_CLASS_BIAS_VAL_PROB,
    axis=1,
)

PAD_DA_VAL_COMPARISON_DF = pd.DataFrame(
    [
        {
            "method": (
                "PAD_DA_NoTTA_StackingBlend"
            ),
            "val_bacc": float(
                balanced_accuracy_score(
                    PAD_DA_Y_VAL,
                    PAD_DA_NO_BIAS_VAL_PRED,
                )
            ),
            "val_macro_f1": float(
                f1_score(
                    PAD_DA_Y_VAL,
                    PAD_DA_NO_BIAS_VAL_PRED,
                    average="macro",
                    zero_division=0,
                )
            ),
            "val_accuracy": float(
                accuracy_score(
                    PAD_DA_Y_VAL,
                    PAD_DA_NO_BIAS_VAL_PRED,
                )
            ),
        },
        {
            "method": (
                "PAD_DA_NoTTA_StackingBlend_ClassBias"
            ),
            "val_bacc": float(
                balanced_accuracy_score(
                    PAD_DA_Y_VAL,
                    PAD_DA_CLASS_BIAS_VAL_PRED,
                )
            ),
            "val_macro_f1": float(
                f1_score(
                    PAD_DA_Y_VAL,
                    PAD_DA_CLASS_BIAS_VAL_PRED,
                    average="macro",
                    zero_division=0,
                )
            ),
            "val_accuracy": float(
                accuracy_score(
                    PAD_DA_Y_VAL,
                    PAD_DA_CLASS_BIAS_VAL_PRED,
                )
            ),
        },
    ]
).sort_values(
    [
        "val_bacc",
        "val_macro_f1",
        "val_accuracy",
    ],
    ascending=[
        False,
        False,
        False,
    ],
).reset_index(drop=True)

PAD_DA_SELECTED_METHOD = str(
    PAD_DA_VAL_COMPARISON_DF.iloc[
        0
    ]["method"]
)

print("=" * 92)
print("PAD DOMAIN-ADAPTED VARIANT VALIDATION COMPARISON")
print("=" * 92)
print(
    "Selected PAD class bias:",
    dict(
        zip(
            PAD_DA_CLASS_NAMES,
            [
                float(value)
                for value in PAD_DA_CLASS_BIAS
            ],
        )
    ),
)
display(
    PAD_DA_VAL_COMPARISON_DF
)
print(
    "PAD validation-selected adapted method:",
    PAD_DA_SELECTED_METHOD,
)


PAD DOMAIN-ADAPTED VARIANT VALIDATION COMPARISON
Selected PAD class bias: {'MEL': -0.20000000298023224, 'NV': 0.20000000298023224, 'BCC': 0.0, 'AK': -0.05000000074505806, 'BKL': 0.0, 'SCC': -0.30000001192092896}


                                 method  val_bacc  val_macro_f1  val_accuracy
0  PAD_DA_NoTTA_StackingBlend_ClassBias  0.551223      0.474553      0.550725
1            PAD_DA_NoTTA_StackingBlend  0.519407      0.448482      0.515942

PAD validation-selected adapted method: PAD_DA_NoTTA_StackingBlend_ClassBias


In [29]:
# ============================================================
# PAD-DA CELL D06: FROZEN TEST EVALUATION OF BOTH ADAPTED MODELS
# ============================================================

raw_pad_da_test_stack = (
    PAD_DA_META_MODEL.predict_proba(
        PAD_DA_META_X_ALL[
            PAD_DA_TEST_INDEX
        ]
    )
)

PAD_DA_TEST_STACK_PROB = (
    pad_da_align_probability(
        raw_pad_da_test_stack,
        PAD_DA_META_MODEL.named_steps[
            "classifier"
        ].classes_,
    )
)

PAD_DA_NO_BIAS_TEST_PROB = (
    normalize_probabilities(
        (
            1.0 - PAD_DA_BEST_ALPHA
        )
        * PAD_DA_ANCHOR_ALL[
            PAD_DA_TEST_INDEX
        ]
        + PAD_DA_BEST_ALPHA
        * PAD_DA_TEST_STACK_PROB
    )
)

PAD_DA_CLASS_BIAS_TEST_PROB = (
    pad_da_apply_bias(
        PAD_DA_NO_BIAS_TEST_PROB,
        PAD_DA_CLASS_BIAS,
    )
)


def pad_da_evaluate(
    y_true,
    probability,
    method_name,
    result_type,
):
    probability = normalize_probabilities(
        probability
    )

    prediction = np.argmax(
        probability,
        axis=1,
    )

    confusion = confusion_matrix(
        y_true,
        prediction,
        labels=np.arange(
            PAD_DA_NUM_CLASSES
        ),
    )

    y_binary = label_binarize(
        y_true,
        classes=np.arange(
            PAD_DA_NUM_CLASSES
        ),
    )

    try:
        auc = float(
            roc_auc_score(
                y_binary,
                probability,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        auc = float("nan")

    summary = {
        "method": method_name,
        "result_type": result_type,
        "n_samples": int(
            len(y_true)
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                prediction,
            )
        ),
        "bacc": float(
            balanced_accuracy_score(
                y_true,
                prediction,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                prediction,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                prediction,
                average="weighted",
                zero_division=0,
            )
        ),
        "auc_macro_ovr": auc,
    }

    report = classification_report(
        y_true,
        prediction,
        labels=np.arange(
            PAD_DA_NUM_CLASSES
        ),
        target_names=(
            PAD_DA_CLASS_NAMES
        ),
        output_dict=True,
        zero_division=0,
    )

    return {
        "summary": summary,
        "prediction": prediction,
        "confusion": confusion,
        "report": report,
        "probability": probability,
    }


# ------------------------------------------------------------
# Same 345-image test-subset zero-shot references
# ------------------------------------------------------------
PAD_DA_ZERO_SHOT_NO_BIAS_ALL = (
    pad_da_restrict_to_six(
        PAD_DA_ZERO_SHOT_ENSEMBLES_8[
            "NoTTA_StackingBlend"
        ]
    )
)

PAD_DA_ZERO_SHOT_CLASS_BIAS_ALL = (
    pad_da_restrict_to_six(
        PAD_DA_ZERO_SHOT_ENSEMBLES_8[
            "NoTTA_StackingBlend_ClassBias"
        ]
    )
)

pad_da_result_objects = OrderedDict(
    [
        (
            "ZeroShot_NoTTA_StackingBlend",
            pad_da_evaluate(
                PAD_DA_Y_TEST,
                PAD_DA_ZERO_SHOT_NO_BIAS_ALL[
                    PAD_DA_TEST_INDEX
                ],
                "ZeroShot_NoTTA_StackingBlend",
                "Zero-shot on PAD adapted-test subset",
            ),
        ),
        (
            "ZeroShot_NoTTA_StackingBlend_ClassBias",
            pad_da_evaluate(
                PAD_DA_Y_TEST,
                PAD_DA_ZERO_SHOT_CLASS_BIAS_ALL[
                    PAD_DA_TEST_INDEX
                ],
                (
                    "ZeroShot_NoTTA_"
                    "StackingBlend_ClassBias"
                ),
                "Zero-shot on PAD adapted-test subset",
            ),
        ),
        (
            "PAD_DA_NoTTA_StackingBlend",
            pad_da_evaluate(
                PAD_DA_Y_TEST,
                PAD_DA_NO_BIAS_TEST_PROB,
                "PAD_DA_NoTTA_StackingBlend",
                "PAD fine-tuned/domain-adapted test",
            ),
        ),
        (
            "PAD_DA_NoTTA_StackingBlend_ClassBias",
            pad_da_evaluate(
                PAD_DA_Y_TEST,
                PAD_DA_CLASS_BIAS_TEST_PROB,
                (
                    "PAD_DA_NoTTA_"
                    "StackingBlend_ClassBias"
                ),
                "PAD fine-tuned/domain-adapted test",
            ),
        ),
    ]
)

PAD_DA_TEST_RESULTS_DF = pd.DataFrame(
    [
        result["summary"]
        for result in (
            pad_da_result_objects.values()
        )
    ]
)

# Add the previous FusionConcat PAD adapter as a reference
# when its saved metric file is available.
previous_pad_metrics_csv = (
    PAD_DA_PREVIOUS_DIR
    / "pad_six_class_test_metrics.csv"
)

if previous_pad_metrics_csv.exists():
    previous_metrics_df = pd.read_csv(
        previous_pad_metrics_csv
    )

    if len(previous_metrics_df) > 0:
        previous_row = (
            previous_metrics_df.iloc[
                0
            ].to_dict()
        )

        def pad_da_previous_metric(
            *names,
        ):
            for name in names:
                if name in previous_row:
                    try:
                        return float(
                            previous_row[name]
                        )
                    except Exception:
                        pass
            return np.nan

        PAD_DA_TEST_RESULTS_DF = pd.concat(
            [
                PAD_DA_TEST_RESULTS_DF,
                pd.DataFrame(
                    [
                        {
                            "method": (
                                "Previous_PAD_FineTuned_"
                                "FusionConcat_Adapter"
                            ),
                            "result_type": (
                                "Previous PAD fine-tuned/"
                                "domain-adapted test reference"
                            ),
                            "n_samples": int(
                                pad_da_previous_metric(
                                    "n_samples",
                                    "samples",
                                )
                                if not np.isnan(
                                    pad_da_previous_metric(
                                        "n_samples",
                                        "samples",
                                    )
                                )
                                else len(
                                    PAD_DA_Y_TEST
                                )
                            ),
                            "accuracy": (
                                pad_da_previous_metric(
                                    "accuracy"
                                )
                            ),
                            "bacc": (
                                pad_da_previous_metric(
                                    "bacc",
                                    "balanced_accuracy",
                                )
                            ),
                            "macro_precision": (
                                pad_da_previous_metric(
                                    "macro_precision"
                                )
                            ),
                            "macro_recall": (
                                pad_da_previous_metric(
                                    "macro_recall"
                                )
                            ),
                            "macro_f1": (
                                pad_da_previous_metric(
                                    "macro_f1"
                                )
                            ),
                            "weighted_f1": (
                                pad_da_previous_metric(
                                    "weighted_f1"
                                )
                            ),
                            "auc_macro_ovr": (
                                pad_da_previous_metric(
                                    "auc_macro_ovr",
                                    "auc",
                                )
                            ),
                        }
                    ]
                ),
            ],
            ignore_index=True,
        )

PAD_DA_TEST_RESULTS_DF = (
    PAD_DA_TEST_RESULTS_DF.sort_values(
        [
            "bacc",
            "macro_f1",
            "accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    ).reset_index(drop=True)
)

PAD_DA_TEST_RESULTS_DF[
    "rank_by_test_bacc"
] = np.arange(
    1,
    len(
        PAD_DA_TEST_RESULTS_DF
    ) + 1,
)

print("=" * 92)
print("PAD FINE-TUNED / DOMAIN-ADAPTED TEST RESULTS")
print("=" * 92)
display(
    PAD_DA_TEST_RESULTS_DF
)

print(
    "\nValidation-selected PAD adapted method:",
    PAD_DA_SELECTED_METHOD,
)

selected_test_row = (
    PAD_DA_TEST_RESULTS_DF.loc[
        PAD_DA_TEST_RESULTS_DF[
            "method"
        ] == PAD_DA_SELECTED_METHOD
    ].iloc[0]
)

print(
    "Selected method PAD test BACC:",
    f"{100.0 * selected_test_row['bacc']:.4f}%",
)


PAD FINE-TUNED / DOMAIN-ADAPTED TEST RESULTS


                                        method  \
0  Previous_PAD_FineTuned_FusionConcat_Adapter   
1                   PAD_DA_NoTTA_StackingBlend   
2         PAD_DA_NoTTA_StackingBlend_ClassBias   
3       ZeroShot_NoTTA_StackingBlend_ClassBias   
4                 ZeroShot_NoTTA_StackingBlend   

                                         result_type  n_samples  accuracy  \
0  Previous PAD fine-tuned/domain-adapted test re...        345  0.689855   
1                 PAD fine-tuned/domain-adapted test        345  0.527536   
2                 PAD fine-tuned/domain-adapted test        345  0.533333   
3               Zero-shot on PAD adapted-test subset        345  0.307246   
4               Zero-shot on PAD adapted-test subset        345  0.304348   

       bacc  macro_precision  macro_recall  macro_f1  weighted_f1  \
0  0.608252         0.653708      0.608252  0.616420     0.678759   
1  0.547127         0.450807      0.547127  0.461130     0.532081   
2  0.543817         0.450081 


Validation-selected PAD adapted method: PAD_DA_NoTTA_StackingBlend_ClassBias
Selected method PAD test BACC: 54.3817%


In [30]:
# ============================================================
# PAD-DA CELL D07: SAVE DETAILED RESULTS AND CREATE ZIP
# ============================================================

PAD_DA_TEST_RESULTS_DF.to_csv(
    PAD_DA_OUTPUT_DIR
    / "PAD_DOMAIN_ADAPTED_BOTH_STACKING_RESULTS.csv",
    index=False,
)

PAD_DA_VAL_COMPARISON_DF.to_csv(
    PAD_DA_OUTPUT_DIR
    / "PAD_DOMAIN_ADAPTED_VALIDATION_SELECTION.csv",
    index=False,
)

PAD_DA_CONFIG = {
    "adaptation_level": (
        "six-class probability/meta-classifier domain adaptation"
    ),
    "base_encoders_retrained": False,
    "base_fusion_models_retrained": False,
    "split_reused_from_previous_pad_adapter": bool(
        PAD_DA_USED_PREVIOUS_SPLIT
    ),
    "split_counts": {
        "train": int(
            len(PAD_DA_TRAIN_INDEX)
        ),
        "validation": int(
            len(PAD_DA_VAL_INDEX)
        ),
        "test": int(
            len(PAD_DA_TEST_INDEX)
        ),
    },
    "common_classes": (
        PAD_DA_CLASS_NAMES
    ),
    "base_models": (
        PAD_DA_MODEL_NAMES
    ),
    "pad_temperatures": {
        model_name: float(
            temperature
        )
        for model_name, temperature
        in PAD_DA_TEMPERATURES.items()
    },
    "selected_C": float(
        PAD_DA_BEST_C
    ),
    "selected_alpha": float(
        PAD_DA_BEST_ALPHA
    ),
    "selected_class_bias": {
        class_name: float(
            bias
        )
        for class_name, bias
        in zip(
            PAD_DA_CLASS_NAMES,
            PAD_DA_CLASS_BIAS,
        )
    },
    "validation_selected_method": (
        PAD_DA_SELECTED_METHOD
    ),
    "pad_test_used_for_selection": False,
}

with open(
    PAD_DA_OUTPUT_DIR
    / "pad_domain_adaptation_config.json",
    "w",
) as file:
    json.dump(
        PAD_DA_CONFIG,
        file,
        indent=2,
    )

for method_name, result in (
    pad_da_result_objects.items()
):
    method_dir = (
        PAD_DA_OUTPUT_DIR
        / method_name
    )
    method_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        result["confusion"],
        index=[
            f"true_{name}"
            for name
            in PAD_DA_CLASS_NAMES
        ],
        columns=[
            f"pred_{name}"
            for name
            in PAD_DA_CLASS_NAMES
        ],
    ).to_csv(
        method_dir
        / "confusion_matrix.csv"
    )

    pd.DataFrame(
        result["report"]
    ).transpose().to_csv(
        method_dir
        / "classification_report.csv"
    )

    prediction_df = pad_da_df.iloc[
        PAD_DA_TEST_INDEX
    ].copy().reset_index(
        drop=True
    )

    prediction_df[
        "y_true_6"
    ] = PAD_DA_Y_TEST

    prediction_df[
        "y_pred_6"
    ] = result["prediction"]

    prediction_df[
        "y_true_name"
    ] = [
        PAD_DA_CLASS_NAMES[
            class_id
        ]
        for class_id
        in PAD_DA_Y_TEST
    ]

    prediction_df[
        "y_pred_name"
    ] = [
        PAD_DA_CLASS_NAMES[
            class_id
        ]
        for class_id
        in result["prediction"]
    ]

    prediction_df[
        "confidence"
    ] = np.max(
        result["probability"],
        axis=1,
    )

    for class_id, class_name in enumerate(
        PAD_DA_CLASS_NAMES
    ):
        prediction_df[
            f"prob_{class_id}_{class_name}"
        ] = result[
            "probability"
        ][
            :,
            class_id,
        ]

    prediction_df.to_csv(
        method_dir
        / "test_predictions.csv",
        index=False,
    )

    with open(
        method_dir
        / "test_metrics.json",
        "w",
    ) as file:
        json.dump(
            result["summary"],
            file,
            indent=2,
        )

np.savez_compressed(
    PAD_DA_OUTPUT_DIR
    / "pad_domain_adapted_probabilities.npz",
    y_train=PAD_DA_Y_TRAIN,
    y_val=PAD_DA_Y_VAL,
    y_test=PAD_DA_Y_TEST,
    no_bias_val_prob=(
        PAD_DA_NO_BIAS_VAL_PROB
    ),
    class_bias_val_prob=(
        PAD_DA_CLASS_BIAS_VAL_PROB
    ),
    no_bias_test_prob=(
        PAD_DA_NO_BIAS_TEST_PROB
    ),
    class_bias_test_prob=(
        PAD_DA_CLASS_BIAS_TEST_PROB
    ),
)

PAD_DA_ARCHIVE_PATH = (
    shutil.make_archive(
        base_name=str(
            Path("/kaggle/working")
            / "pad_domain_adapted_both_stacking_models"
        ),
        format="zip",
        root_dir=str(
            PAD_DA_OUTPUT_DIR
        ),
    )
)

print("=" * 92)
print("PAD DOMAIN ADAPTATION COMPLETED")
print("=" * 92)
print(
    "Results:",
    PAD_DA_OUTPUT_DIR
    / "PAD_DOMAIN_ADAPTED_BOTH_STACKING_RESULTS.csv",
)
print(
    "ZIP:",
    PAD_DA_ARCHIVE_PATH,
)
print(
    "\nReport these results as PAD fine-tuned/domain-adapted, "
    "not as zero-shot external evaluation."
)


PAD DOMAIN ADAPTATION COMPLETED
Results: /kaggle/working/pad_domain_adapted_both_stacking_models/PAD_DOMAIN_ADAPTED_BOTH_STACKING_RESULTS.csv
ZIP: /kaggle/working/pad_domain_adapted_both_stacking_models.zip

Report these results as PAD fine-tuned/domain-adapted, not as zero-shot external evaluation.
